# LeetCode 239 — Sliding Window Maximum (Hard) bằng Python

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxSlidingWindow(self, nums: List[int], k: int) -> List[int]:

```

Đây là bài **Hard** kinh điển về **cửa sổ trượt** (sliding window), và cũng là "bài nhập môn" của một cấu trúc rất đẹp: **monotonic deque** (hàng đợi hai đầu đơn điệu). Đề chỉ có một câu: *"trả về giá trị lớn nhất của mọi cửa sổ độ dài `k`"*. Cách ngây thơ thì ai cũng nghĩ ra trong một phút, nhưng với `n = 10^5` và `k` lớn thì nó chạy quá chậm.

Notebook này đi **từng bậc thang**. Ở mỗi bậc ta trả lời hai câu: *cách cũ vướng ở đâu?* và *mấu chốt nào giúp vượt qua?*

---

## Sau notebook này, bạn sẽ nắm được

1. Hình dung "cửa sổ trượt" bằng hình vẽ, biết vì sao có đúng `n - k + 1` cửa sổ, dùng thành thạo slicing `nums[s:s+k]`, `range`, `enumerate`.
2. **Cách 1 — Brute force**: gọi `max()` cho từng cửa sổ, `O(n·k)`. Kèm bản cải tiến **1b** "giữ max hiện tại, chỉ quét lại khi max rời cửa sổ" — vẫn `O(n·k)` ở worst case, nhưng chỉ ra đúng thứ ta còn thiếu: **ứng viên kế tiếp**.
3. **Cách 2 — Max-heap + lazy deletion** (xoá lười): học `heapq` từ đầu, mẹo đổi dấu, tuple `(-giá trị, chỉ số)`; `O(n log n)`.
4. **Cách 3 — Chia khối kích thước `k` + prefix max / suffix max**: một cách sáng tạo chỉ dùng mảng, `O(n)`.
5. **Cách 4 — Monotonic deque** (cách chính): học `collections.deque` từ căn bản, hiểu vì sao được phép **vứt bỏ vĩnh viễn** một số phần tử, trace bảng từng bước, hiểu phân tích **amortized** (khấu hao) `O(n)`.
6. Code nộp LeetCode, kiểm thử đối chiếu chéo với brute force, lỗi hay gặp, mẹo nhận dạng pattern, 6 bài tập (có LeetCode 1438 và 1696) và đáp án ở cuối.

## 1. Đề bài (theo LeetCode)

Cho mảng số nguyên `nums`. Có một **cửa sổ trượt** (sliding window) kích thước `k` di chuyển từ **tận cùng bên trái** của mảng sang **tận cùng bên phải**. Bạn chỉ nhìn thấy `k` số nằm trong cửa sổ. Mỗi lần, cửa sổ trượt sang phải **đúng một vị trí**.

Hãy trả về *max sliding window* — tức là **danh sách giá trị lớn nhất của từng cửa sổ**, theo thứ tự từ trái sang phải.

**Example 1**

```text
Input:  nums = [1,3,-1,-3,5,3,6,7], k = 3
Output: [3,3,5,5,6,7]
Explanation:
Window position                Max
---------------               -----
[1  3  -1] -3  5  3  6  7       3
 1 [3  -1  -3] 5  3  6  7       3
 1  3 [-1  -3  5] 3  6  7       5
 1  3  -1 [-3  5  3] 6  7       5
 1  3  -1  -3 [5  3  6] 7       6
 1  3  -1  -3  5 [3  6  7]      7
```

**Example 2**

```text
Input:  nums = [1], k = 1
Output: [1]
```

**Constraints**

- `1 <= nums.length <= 10^5`
- `-10^4 <= nums[i] <= 10^4`
- `1 <= k <= nums.length`

Vài điều cần để ý ngay từ đầu (gọi `n = len(nums)`):

- Kết quả là **một list** có đúng `n - k + 1` phần tử — mỗi cửa sổ cho một giá trị max.
- Constraints bảo đảm `1 <= k <= n`, nên **luôn có ít nhất một cửa sổ**; không cần lo mảng rỗng hay `k` lớn hơn `n`.
- Có **số âm** và có thể có **số trùng nhau** — hai thứ hay làm code sai một cách âm thầm.
- `n` tới `10^5` và `k` có thể tới `n`. Cách "mỗi cửa sổ quét `k` phần tử" tốn khoảng `(n - k + 1) · k` bước; với `k = n/2` là khoảng `2.5 · 10^9` bước → chắc chắn **TLE** (Time Limit Exceeded — quá thời gian). Ta cần `O(n)` hoặc `O(n log n)`.

In [1]:
from typing import List   # LeetCode tự import sẵn; chạy ngoài LeetCode thì ta tự import

nums1, k1 = [1, 3, -1, -3, 5, 3, 6, 7], 3
nums2, k2 = [1], 1

print("Example 1:", nums1, "k =", k1, "-> mong đợi [3, 3, 5, 5, 6, 7]")
print("Example 2:", nums2, "k =", k2, "-> mong đợi [1]")
print("Số cửa sổ của example 1: n - k + 1 =", len(nums1), "-", k1, "+ 1 =", len(nums1) - k1 + 1)

Example 1: [1, 3, -1, -3, 5, 3, 6, 7] k = 3 -> mong đợi [3, 3, 5, 5, 6, 7]
Example 2: [1] k = 1 -> mong đợi [1]
Số cửa sổ của example 1: n - k + 1 = 8 - 3 + 1 = 6


## 2. Kiến thức nền: cửa sổ, slicing, `range` và `enumerate`

### Cửa sổ là gì, viết bằng chỉ số thế nào?

Một **cửa sổ** độ dài `k` là `k` phần tử **liền nhau**. Nếu cửa sổ **bắt đầu** ở chỉ số `s` thì nó gồm các chỉ số

```text
s, s+1, s+2, ..., s+k-1
```

Ví dụ `nums = [1,3,-1,-3,5,3,6,7]`, `k = 3`, `s = 2` → chỉ số `2, 3, 4` → giá trị `[-1, -3, 5]`.

Khi cửa sổ **trượt sang phải một bước** (`s` tăng 1): phần tử `nums[s]` **rời đi** ở bên trái, phần tử `nums[s+k]` **đi vào** ở bên phải, còn `k - 1` phần tử ở giữa **giữ nguyên**. Ghi nhớ chi tiết này — nó là chìa khoá của mọi cách tối ưu phía sau.

Nhiều lúc ta sẽ gọi cửa sổ theo **chỉ số kết thúc** `i` thay vì chỉ số bắt đầu: cửa sổ kết thúc tại `i` gồm `i-k+1, ..., i`. Hai cách gọi chỉ lệch nhau `i = s + k - 1`.

### Slicing `nums[a:b]` — lấy từ `a` tới **trước** `b`

`nums[a:b]` lấy các phần tử có chỉ số `a, a+1, ..., b-1` (**không** lấy `b`). Vì vậy `nums[s:s+k]` lấy đúng `k` phần tử của cửa sổ bắt đầu ở `s`.

### Có bao nhiêu cửa sổ? Vì sao là `range(n - k + 1)`?

Cửa sổ cuối cùng phải kết thúc ở chỉ số `n - 1`, tức bắt đầu ở `s = n - k`. Vậy `s` chạy `0, 1, ..., n - k` → có `n - k + 1` giá trị. Trong Python, `range(n - k + 1)` sinh đúng các số `0..n-k` (range cũng "không lấy đầu mút phải", giống slicing).

Kiểm tra bằng số: `n = 8, k = 3` → `s = 0..5` → **6** cửa sổ (đúng 6 dòng trong Explanation của đề). Hai trường hợp biên: `k = n` → 1 cửa sổ; `k = 1` → `n` cửa sổ (mỗi phần tử là một cửa sổ).

### `enumerate` — lấy cả chỉ số lẫn giá trị

`for i, x in enumerate(nums)` cho ta lần lượt `(0, nums[0]), (1, nums[1]), ...`. Các cách 2 và 4 cần **chỉ số** để biết một phần tử đã rời cửa sổ chưa, nên ta sẽ dùng `enumerate` rất nhiều.

In [2]:
nums = [1, 3, -1, -3, 5, 3, 6, 7]
k = 3
n = len(nums)

print("nums[2:5]        =", nums[2:5], "  <- chỉ số 2, 3, 4 (không lấy 5)")
print("range(n - k + 1) =", list(range(n - k + 1)), "  <- các vị trí bắt đầu s")
print("cửa sổ cuối      =", nums[n - k:n], "  <- bắt đầu ở s = n - k =", n - k)

print("\nTrượt từ s = 0 sang s = 1:")
print("  rời đi:", nums[0], "| đi vào:", nums[0 + k], "| giữ nguyên:", nums[1:k])

print("\nenumerate:")
for i, x in enumerate(nums[:4]):
    print("  i =", i, "x =", x)

nums[2:5]        = [-1, -3, 5]   <- chỉ số 2, 3, 4 (không lấy 5)
range(n - k + 1) = [0, 1, 2, 3, 4, 5]   <- các vị trí bắt đầu s
cửa sổ cuối      = [3, 6, 7]   <- bắt đầu ở s = n - k = 5

Trượt từ s = 0 sang s = 1:
  rời đi: 1 | đi vào: -3 | giữ nguyên: [3, -1]

enumerate:
  i = 0 x = 1
  i = 1 x = 3
  i = 2 x = -1
  i = 3 x = -3


### Vẽ các cửa sổ ra cho dễ thấy

Hàm `show_windows(nums, k)` dưới đây in mọi cửa sổ giống bảng Explanation của LeetCode: cửa sổ được đóng trong ngoặc `[ ]`, cột cuối là max. Bạn **không cần** hiểu hết chi tiết định dạng chuỗi trong hàm; chỉ cần biết cách gọi.

Ngoài ví dụ của đề, notebook dùng xuyên suốt một **ví dụ thứ hai** `nums = [9,5,7,3,1,4,4,2], k = 3`. Nó được chọn có chủ đích:

- số lớn nhất `9` nằm ngay đầu và phải **rời cửa sổ** khi còn đang là max — tình huống bẫy của nhiều cách giải;
- có hai số **bằng nhau** `4, 4` liền nhau — tình huống bẫy thứ hai.

In [3]:
def show_windows(nums, k):
    w = max(len(str(x)) for x in nums)          # độ rộng mỗi số để căn cột
    for s in range(len(nums) - k + 1):
        line = ""
        for j, x in enumerate(nums):
            left = "[" if j == s else " "
            right = "]" if j == s + k - 1 else " "
            line += left + f"{x:>{w}}" + right
        print(f"s={s}: {line}   max = {max(nums[s:s + k])}")

nums_ex2, k_ex2 = [9, 5, 7, 3, 1, 4, 4, 2], 3

print("Example 1, k = 3")
show_windows(nums1, k1)
print("\nVí dụ thứ hai, k = 3")
show_windows(nums_ex2, k_ex2)

Example 1, k = 3
s=0: [ 1   3  -1] -3   5   3   6   7    max = 3
s=1:   1 [ 3  -1  -3]  5   3   6   7    max = 3
s=2:   1   3 [-1  -3   5]  3   6   7    max = 5
s=3:   1   3  -1 [-3   5   3]  6   7    max = 5
s=4:   1   3  -1  -3 [ 5   3   6]  7    max = 6
s=5:   1   3  -1  -3   5 [ 3   6   7]   max = 7

Ví dụ thứ hai, k = 3
s=0: [9  5  7] 3  1  4  4  2    max = 9
s=1:  9 [5  7  3] 1  4  4  2    max = 7
s=2:  9  5 [7  3  1] 4  4  2    max = 7
s=3:  9  5  7 [3  1  4] 4  2    max = 4
s=4:  9  5  7  3 [1  4  4] 2    max = 4
s=5:  9  5  7  3  1 [4  4  2]   max = 4


## 3. Cách 1 — Brute force: hỏi `max()` cho từng cửa sổ

Cách người mới nghĩ ra đầu tiên, và cũng là cách **nên** viết đầu tiên vì nó gần như không thể sai:

```text
với mỗi vị trí bắt đầu s = 0, 1, ..., n-k:
    lấy cửa sổ nums[s : s+k]
    tìm max của nó, thêm vào kết quả
```

Hàm có sẵn `max(list)` của Python làm đúng việc "tìm max". Nhưng đừng nghĩ `max()` là miễn phí: bên trong nó vẫn phải **nhìn qua từng phần tử** — nhớ "lớn nhất tạm thời", gặp số lớn hơn thì thay. Với cửa sổ `k` phần tử, nó cần `k - 1` phép so sánh.

Để thấy rõ chi phí đó, cell dưới tự viết lại `max` bằng vòng lặp và **đếm số phép so sánh**.

In [4]:
def max_by_hand(window):
    best = window[0]
    comparisons = 0
    for x in window[1:]:
        comparisons += 1
        if x > best:
            best = x
    return best, comparisons

print(max_by_hand([1, 3, -1]))          # (3, 2): max = 3, tốn 2 phép so sánh
print(max_by_hand([-3, 5, 3, 6, 7]))    # (7, 4)
print(max([1, 3, -1]))                  # hàm có sẵn cho cùng kết quả, chỉ là viết bằng C nên nhanh hơn

(3, 2)
(7, 4)
3


In [5]:
def max_sliding_window_bruteforce(nums: List[int], k: int) -> List[int]:
    result = []
    for s in range(len(nums) - k + 1):
        result.append(max(nums[s:s + k]))
    return result

print(max_sliding_window_bruteforce(nums1, k1))          # [3, 3, 5, 5, 6, 7]
print(max_sliding_window_bruteforce(nums2, k2))          # [1]
print(max_sliding_window_bruteforce(nums_ex2, k_ex2))    # [9, 7, 7, 4, 4, 4]

[3, 3, 5, 5, 6, 7]
[1]
[9, 7, 7, 4, 4, 4]


### Độ phức tạp cách 1 và chỗ vướng

Gọi `n = len(nums)`.

- Có `n - k + 1` cửa sổ, mỗi cửa sổ tốn `O(k)` (slicing copy `k` phần tử + `max` quét `k` phần tử).
- **Time:** `O((n - k + 1) · k)`, thường viết gọn là `O(n·k)`.
- **Space:** `O(k)` cho bản copy cửa sổ (không tính list kết quả).

Tích `(n - k + 1) · k` **nhỏ** khi `k` rất nhỏ hoặc `k` gần bằng `n`, và **lớn nhất** khi `k ≈ n/2`: khoảng `n²/4`. Với `n = 10^5` là `2.5 · 10^9` — quá chậm (cell dưới tính cụ thể).

**Lãng phí nằm ở đâu?** Hai cửa sổ liên tiếp **chung nhau `k - 1` phần tử**. Ví dụ `[1, 3, -1]` và `[3, -1, -3]` chung `[3, -1]`. Brute force quên sạch những gì vừa biết và quét lại `3, -1` từ đầu. Mọi cách tối ưu phía sau đều trả lời một câu: **làm sao tái sử dụng thông tin của cửa sổ trước?**

In [6]:
n = 10**5
print(f"{'k':>7} | số bước ≈ (n - k + 1) * k")
for k in [1, 10, 1000, 50000, 99000, 100000]:
    print(f"{k:>7} | {(n - k + 1) * k:>15,}")

a, b = nums1[0:3], nums1[1:4]
print("\nHai cửa sổ liên tiếp:", a, "và", b, "-> chung nhau", a[1:], "(k - 1 = 2 phần tử)")

      k | số bước ≈ (n - k + 1) * k
      1 |         100,000
     10 |         999,910
   1000 |      99,001,000
  50000 |   2,500,050,000
  99000 |      99,099,000
 100000 |         100,000

Hai cửa sổ liên tiếp: [1, 3, -1] và [3, -1, -3] -> chung nhau [3, -1] (k - 1 = 2 phần tử)


### Cách 1b — Giữ "max hiện tại", chỉ quét lại khi thật sự cần

Cải tiến tự nhiên đầu tiên: đừng tính lại từ đầu, hãy **cập nhật** max khi cửa sổ trượt. Mỗi bước có một phần tử `entering` đi vào và một phần tử `leaving` rời đi. Gọi `cur_max` là max của cửa sổ cũ. Có 3 trường hợp:

1. `entering >= cur_max` → max mới **chắc chắn** là `entering`. Vì sao? Mọi phần tử của cửa sổ mới hoặc nằm trong cửa sổ cũ (nên `<= cur_max <= entering`), hoặc chính là `entering`.
2. `entering < cur_max` và `leaving != cur_max` → `leaving` nhỏ hơn `cur_max`, nên phần tử đạt `cur_max` **vẫn còn** trong cửa sổ → max giữ nguyên `cur_max`.
3. `entering < cur_max` và `leaving == cur_max` → có thể max vừa **rời đi**. Ta không biết max mới là ai → đành **quét lại** cả cửa sổ.

Trường hợp 3 là "điểm đau". Trường hợp 1 và 2 chỉ tốn `O(1)`.

Ví dụ thứ hai `[9,5,7,3,1,4,4,2]`, `k = 3`: khi `9` rời đi (phần tử vào là `3 < 9`), ta phải quét lại cả cửa sổ `[5, 7, 3]` mới biết max mới là `7`; khi `7` rời đi (phần tử vào là `4 < 7`), lại phải quét lại `[3, 1, 4]` để ra `4`. Hàm dưới đếm số lần quét lại.

In [7]:
def max_sliding_window_rescan(nums: List[int], k: int):
    cur_max = max(nums[:k])
    result = [cur_max]
    rescans = 0
    for i in range(k, len(nums)):          # i = chỉ số phần tử đi vào
        entering, leaving = nums[i], nums[i - k]
        if entering >= cur_max:            # trường hợp 1
            cur_max = entering
        elif leaving == cur_max:           # trường hợp 3: max có thể vừa rời đi
            cur_max = max(nums[i - k + 1:i + 1])
            rescans += 1
        # trường hợp 2: không làm gì, cur_max giữ nguyên
        result.append(cur_max)
    return result, rescans

print(max_sliding_window_rescan(nums1, k1))          # ([3, 3, 5, 5, 6, 7], 0)  -> không phải quét lại lần nào
print(max_sliding_window_rescan(nums_ex2, k_ex2))    # ([9, 7, 7, 4, 4, 4], 2)  -> quét lại 2 lần

([3, 3, 5, 5, 6, 7], 0)
([9, 7, 7, 4, 4, 4], 2)


### Cách 1b vướng ở đâu? Mảng **giảm dần**

Với mảng giảm dần như `[10, 9, 8, 7, ...]`, max của mỗi cửa sổ luôn là **phần tử đầu tiên** — đúng phần tử sẽ rời đi ở bước sau. Vậy **bước nào cũng rơi vào trường hợp 3**, bước nào cũng quét lại `k` phần tử → worst case vẫn là `O(n·k)`, không hơn brute force.

Nhưng cách 1b cho ta một **câu hỏi rất đúng**: *"Khi max rời cửa sổ, max mới là ai?"* Nếu ta **nhớ sẵn danh sách các ứng viên kế tiếp** thì không cần quét lại. Có ba hướng trả lời, cũng là ba cách tiếp theo của notebook:

- **Cách 2:** giữ **tất cả** phần tử trong một cấu trúc luôn cho biết "ai lớn nhất" nhanh → **heap**.
- **Cách 3:** **tính trước** các max cục bộ bằng mảng, rồi ghép lại → **chia khối + prefix/suffix max**.
- **Cách 4:** chỉ giữ những phần tử **còn hy vọng** làm max, theo đúng thứ tự "ai sẽ là max kế tiếp" → **monotonic deque**.

In [8]:
def count_rescans(nums, k):
    return max_sliding_window_rescan(nums, k)[1]

n, k = 2000, 1000
decreasing = list(range(n, 0, -1))        # [2000, 1999, ..., 1]
print("Mảng giảm dần: số lần quét lại =", count_rescans(decreasing, k), "trên", n - k, "bước trượt")
print("  -> tổng số phần tử phải quét lại ≈", count_rescans(decreasing, k) * k)

import random
random.seed(239)
rand_arr = [random.randint(-10**4, 10**4) for _ in range(n)]
print("Mảng ngẫu nhiên: số lần quét lại =", count_rescans(rand_arr, k), "(ít, nhưng không đảm bảo được cho mọi input)")

Mảng giảm dần: số lần quét lại = 1000 trên 1000 bước trượt
  -> tổng số phần tử phải quét lại ≈ 1000000
Mảng ngẫu nhiên: số lần quét lại = 0 (ít, nhưng không đảm bảo được cho mọi input)


## 4. Cách 2 — Max-heap + lazy deletion

Ta cần một "cái túi" chứa các phần tử của cửa sổ, hỗ trợ nhanh ba việc:

1. **thêm** phần tử mới đi vào,
2. **xem** phần tử lớn nhất,
3. **bỏ** phần tử lớn nhất (khi nó đã rời cửa sổ).

Cấu trúc dữ liệu sinh ra cho đúng ba việc này là **heap**.

### Kiến thức nền: heap (đống) và module `heapq`

**Heap** (tiếng Việt hay gọi là *đống*) trả lời rất nhanh câu hỏi *"phần tử nhỏ nhất hiện tại là gì?"*, kể cả khi ta liên tục thêm/bớt phần tử.

Trong Python, heap chỉ là **một list bình thường** được sắp xếp "một phần" theo quy tắc: nếu nhìn list như một cây nhị phân (phần tử ở vị trí `p` có hai con ở vị trí `2p + 1` và `2p + 2`) thì **cha luôn ≤ con**. Hệ quả: phần tử nhỏ nhất luôn nằm ở gốc, tức `heap[0]`.

Module `heapq` (thư viện chuẩn) cung cấp:

| Lệnh | Ý nghĩa | Thời gian |
|---|---|---|
| `heapq.heappush(h, x)` | thêm `x` vào heap `h` | `O(log m)` |
| `heapq.heappop(h)` | lấy ra **và xoá** phần tử nhỏ nhất | `O(log m)` |
| `h[0]` | **xem** phần tử nhỏ nhất (không xoá) | `O(1)` |

(`m` = số phần tử đang có trong heap.) Vì sao `log m`? Cây có chiều cao khoảng `log₂ m`; thêm/xoá chỉ cần "đẩy" một phần tử đi dọc **một nhánh** từ lá lên gốc (hoặc từ gốc xuống lá) để khôi phục quy tắc cha ≤ con.

Lưu ý quan trọng: heap **không phải** list đã sắp xếp. Chỉ `h[0]` được bảo đảm là nhỏ nhất; các vị trí khác chỉ thoả "cha ≤ con". Cell dưới cho bạn nhìn tận mắt.

In [9]:
import heapq

h = []
for x in [5, 1, 8, 3, 9, 2]:
    heapq.heappush(h, x)
    print(f"push {x}: heap = {h}   nhỏ nhất h[0] = {h[0]}")

print("\nKiểm tra cha <= con:", all(h[p] <= h[c] for c in range(1, len(h)) for p in [(c - 1) // 2]))
print("Pop lần lượt:", [heapq.heappop(h) for _ in range(6)], "<- ra theo thứ tự tăng dần")

push 5: heap = [5]   nhỏ nhất h[0] = 5
push 1: heap = [1, 5]   nhỏ nhất h[0] = 1
push 8: heap = [1, 5, 8]   nhỏ nhất h[0] = 1
push 3: heap = [1, 3, 8, 5]   nhỏ nhất h[0] = 1
push 9: heap = [1, 3, 8, 5, 9]   nhỏ nhất h[0] = 1
push 2: heap = [1, 3, 2, 5, 9, 8]   nhỏ nhất h[0] = 1

Kiểm tra cha <= con: True
Pop lần lượt: [1, 2, 3, 5, 8, 9] <- ra theo thứ tự tăng dần


### Biến min-heap thành max-heap: mẹo **đổi dấu**

`heapq` chỉ có **min-heap**. Ta cần **max**. Mẹo kinh điển: lưu `-x` thay cho `x`. Số lớn nhất có số đối **nhỏ nhất**, nên nó sẽ nằm ở `heap[0]`; lấy ra thì đổi dấu lại: `max = -heap[0]`.

Ví dụ: các số `1, 3, -1` → lưu `-1, -3, 1` → nhỏ nhất là `-3` → max là `3`.

### Tuple trong heap: so sánh **từng thành phần**

Ta còn cần biết phần tử đang ở đỉnh **nằm ở chỉ số nào** (để biết nó đã rời cửa sổ chưa). Vì vậy mỗi phần tử trong heap là một **tuple** `(-giá trị, chỉ số)`.

Python so sánh hai tuple theo kiểu "từ điển": so thành phần đầu trước; chỉ khi bằng nhau mới so tới thành phần thứ hai.

- `(-5, 4) < (-3, 1)` vì `-5 < -3` (giá trị 5 lớn hơn 3 → được ưu tiên).
- `(-4, 5) < (-4, 6)` vì thành phần đầu bằng nhau, xét tiếp `5 < 6`.

Nhờ vậy, đỉnh heap luôn là phần tử **lớn nhất**; nếu bằng nhau thì là phần tử có chỉ số nhỏ hơn.

In [10]:
print((-5, 4) < (-3, 1))   # True: so -5 với -3
print((-4, 5) < (-4, 6))   # True: -4 == -4 nên so tiếp 5 với 6

h = []
for i, x in enumerate([1, 3, -1]):
    heapq.heappush(h, (-x, i))
print("heap =", h)
print("đỉnh =", h[0], "-> max =", -h[0][0], "nằm ở chỉ số", h[0][1])

True
True
heap = [(-3, 1), (-1, 0), (1, 2)]
đỉnh = (-3, 1) -> max = 3 nằm ở chỉ số 1


### Ý tưởng **lazy deletion** (xoá lười)

Vấn đề: khi cửa sổ trượt, phần tử `nums[i-k]` rời đi. Muốn xoá nó khỏi heap thì phải **tìm** nó trong list (`O(m)`) rồi sửa lại heap — chậm và rườm rà.

Mấu chốt: **ta chỉ quan tâm đỉnh heap**. Một phần tử đã rời cửa sổ mà nằm sâu bên dưới thì **vô hại** — nó chỉ gây sai khi nó leo lên đỉnh. Vậy ta *lười*: không xoá ngay, chỉ dọn khi cần.

Mỗi khi cần đọc max:

```text
while chỉ số của đỉnh đã rời cửa sổ:
    heappop  (bỏ đỉnh "chết")
đỉnh còn lại chính là max của cửa sổ
```

Khi nào chỉ số `j` "đã rời cửa sổ"? Cửa sổ kết thúc tại `i` gồm các chỉ số `i-k+1 .. i`. Vậy `j` đã rời đi khi **`j <= i - k`**. Ví dụ `i = 5, k = 3`: cửa sổ là `3..5`, nên `j = 2` đã rời (`2 <= 2`), còn `j = 3` vẫn ở trong.

Thuật toán cho từng `i`:

1. `heappush(heap, (-nums[i], i))`
2. dọn đỉnh: `while heap[0][1] <= i - k: heappop(heap)`
3. nếu `i >= k - 1` (đã đủ `k` phần tử): ghi `-heap[0][0]`

Bước 2 không bao giờ làm heap rỗng: phần tử vừa push có chỉ số `i > i - k` nên không bao giờ bị coi là "chết".

In [11]:
def max_sliding_window_heap(nums: List[int], k: int) -> List[int]:
    heap = []                                  # phần tử: (-giá trị, chỉ số)
    result = []
    for i, x in enumerate(nums):
        heapq.heappush(heap, (-x, i))
        while heap[0][1] <= i - k:             # đỉnh đã rời cửa sổ -> bỏ (lazy deletion)
            heapq.heappop(heap)
        if i >= k - 1:
            result.append(-heap[0][0])
    return result

print(max_sliding_window_heap(nums1, k1))          # [3, 3, 5, 5, 6, 7]
print(max_sliding_window_heap(nums2, k2))          # [1]
print(max_sliding_window_heap(nums_ex2, k_ex2))    # [9, 7, 7, 4, 4, 4]

[3, 3, 5, 5, 6, 7]
[1]
[9, 7, 7, 4, 4, 4]


### Trace cách 2

Ý nghĩa các cột: `push` là tuple vừa thêm; "bỏ" là các đỉnh bị `heappop` vì đã hết hạn; "đỉnh" là `heap[0]` **sau khi dọn**; "size" là số phần tử trong heap; cột cuối đếm số phần tử **đã hết hạn nhưng vẫn nằm trong heap** (rác chưa dọn).

**Example 1** `[1,3,-1,-3,5,3,6,7]`, `k = 3`:

| i | x | push | bỏ (hết hạn) | đỉnh sau khi dọn | max | size | hết hạn mà còn trong heap |
|---:|---:|---|---|---|---:|---:|---:|
| 0 | 1 | `(-1, 0)` | — | `(-1, 0)` | — | 1 | 0 |
| 1 | 3 | `(-3, 1)` | — | `(-3, 1)` | — | 2 | 0 |
| 2 | -1 | `(1, 2)` | — | `(-3, 1)` | **3** | 3 | 0 |
| 3 | -3 | `(3, 3)` | — | `(-3, 1)` | **3** | 4 | 1 |
| 4 | 5 | `(-5, 4)` | — | `(-5, 4)` | **5** | 5 | 2 |
| 5 | 3 | `(-3, 5)` | — | `(-5, 4)` | **5** | 6 | 3 |
| 6 | 6 | `(-6, 6)` | — | `(-6, 6)` | **6** | 7 | 4 |
| 7 | 7 | `(-7, 7)` | — | `(-7, 7)` | **7** | 8 | 5 |

Nhận xét: ở example 1 nhánh "bỏ" **không chạy lần nào**, vì đỉnh lúc nào cũng còn trong cửa sổ. Nhưng rác tích tụ dần: tới `i = 7`, có 5 phần tử đã hết hạn vẫn nằm trong heap. Chúng vô hại vì không bao giờ lên đỉnh — đúng tinh thần "lười".

**Ví dụ thứ hai** `[9,5,7,3,1,4,4,2]`, `k = 3` — nhánh "bỏ" mới lộ diện:

| i | x | push | bỏ (hết hạn) | đỉnh sau khi dọn | max | size | hết hạn mà còn trong heap |
|---:|---:|---|---|---|---:|---:|---:|
| 0 | 9 | `(-9, 0)` | — | `(-9, 0)` | — | 1 | 0 |
| 1 | 5 | `(-5, 1)` | — | `(-9, 0)` | — | 2 | 0 |
| 2 | 7 | `(-7, 2)` | — | `(-9, 0)` | **9** | 3 | 0 |
| 3 | 3 | `(-3, 3)` | `(-9, 0)` | `(-7, 2)` | **7** | 3 | 0 |
| 4 | 1 | `(-1, 4)` | — | `(-7, 2)` | **7** | 4 | 1 |
| 5 | 4 | `(-4, 5)` | `(-7, 2)`, `(-5, 1)` | `(-4, 5)` | **4** | 3 | 0 |
| 6 | 4 | `(-4, 6)` | — | `(-4, 5)` | **4** | 4 | 1 |
| 7 | 2 | `(-2, 7)` | — | `(-4, 5)` | **4** | 5 | 2 |

- `i = 3`: đỉnh `(-9, 0)` hết hạn (`0 <= 3 - 3`) → bỏ; đỉnh mới `(-7, 2)` còn hạn → max 7.
- `i = 5`: bỏ **hai** đỉnh liên tiếp: `(-7, 2)` vừa hết hạn, rồi lộ ra `(-5, 1)` — phần tử này **đã hết hạn từ `i = 4`** nhưng lúc đó nằm dưới nên được "để dành", giờ lên đỉnh mới bị dọn. Đây là lý do bước dọn phải là `while`, không phải `if`.
- `i = 6`: hai số `4` bằng nhau, tuple `(-4, 5) < (-4, 6)` nên đỉnh là `(-4, 5)` — vẫn đúng max 4.

In [12]:
def trace_heap(nums, k):
    heap = []
    result = []
    for i, x in enumerate(nums):
        heapq.heappush(heap, (-x, i))
        removed = []
        while heap[0][1] <= i - k:
            removed.append(heapq.heappop(heap))
        line = f"i={i} x={x:>2} | push {(-x, i)} | bỏ: {removed if removed else '-'} | đỉnh={heap[0]}"
        if i >= k - 1:
            result.append(-heap[0][0])
            line += f" -> max={-heap[0][0]}"
        print(line + f" | size={len(heap)}")
    return result

print("Kết quả:", trace_heap(nums1, k1))
print()
print("Kết quả:", trace_heap(nums_ex2, k_ex2))

i=0 x= 1 | push (-1, 0) | bỏ: - | đỉnh=(-1, 0) | size=1
i=1 x= 3 | push (-3, 1) | bỏ: - | đỉnh=(-3, 1) | size=2
i=2 x=-1 | push (1, 2) | bỏ: - | đỉnh=(-3, 1) -> max=3 | size=3
i=3 x=-3 | push (3, 3) | bỏ: - | đỉnh=(-3, 1) -> max=3 | size=4
i=4 x= 5 | push (-5, 4) | bỏ: - | đỉnh=(-5, 4) -> max=5 | size=5
i=5 x= 3 | push (-3, 5) | bỏ: - | đỉnh=(-5, 4) -> max=5 | size=6
i=6 x= 6 | push (-6, 6) | bỏ: - | đỉnh=(-6, 6) -> max=6 | size=7
i=7 x= 7 | push (-7, 7) | bỏ: - | đỉnh=(-7, 7) -> max=7 | size=8
Kết quả: [3, 3, 5, 5, 6, 7]

i=0 x= 9 | push (-9, 0) | bỏ: - | đỉnh=(-9, 0) | size=1
i=1 x= 5 | push (-5, 1) | bỏ: - | đỉnh=(-9, 0) | size=2
i=2 x= 7 | push (-7, 2) | bỏ: - | đỉnh=(-9, 0) -> max=9 | size=3
i=3 x= 3 | push (-3, 3) | bỏ: [(-9, 0)] | đỉnh=(-7, 2) -> max=7 | size=3
i=4 x= 1 | push (-1, 4) | bỏ: - | đỉnh=(-7, 2) -> max=7 | size=4
i=5 x= 4 | push (-4, 5) | bỏ: [(-7, 2), (-5, 1)] | đỉnh=(-4, 5) -> max=4 | size=3
i=6 x= 4 | push (-4, 6) | bỏ: - | đỉnh=(-4, 5) -> max=4 | size=4
i=7 x= 2 

### Độ phức tạp cách 2 và chỗ vướng

Gọi `n = len(nums)`.

- Mỗi chỉ số được `heappush` **đúng 1 lần** và bị `heappop` **tối đa 1 lần** → tổng cộng không quá `2n` thao tác heap, mỗi thao tác `O(log n)`.
- **Time:** `O(n log n)`.
- **Space:** `O(n)` — vì lười, rác có thể tích tụ. Worst case là mảng **tăng dần**: phần tử mới luôn lớn nhất nên luôn ở đỉnh, các phần tử cũ không bao giờ lên đỉnh nên **không bao giờ bị dọn** → heap chứa cả `n` phần tử.

`O(n log n)` với `n = 10^5` là chấp nhận được trên LeetCode. Nhưng nhìn lại bảng trace, heap đang giữ **rất nhiều thứ vô dụng**:

- phần tử đã hết hạn (rác của lazy deletion),
- và cả những phần tử **chưa hết hạn nhưng không bao giờ có cơ hội làm max nữa**. Ví dụ ở example 1, khi `3` (chỉ số 1) đi vào, số `1` (chỉ số 0) trở nên vô dụng vĩnh viễn: `3` lớn hơn **và** ở lại cửa sổ lâu hơn. Vậy mà heap vẫn ôm `1`.

Ý tưởng "vứt ngay những phần tử vô dụng" sẽ dẫn tới cách 4. Nhưng trước đó, hãy xem một cách `O(n)` rất khéo **chỉ dùng mảng**.

In [13]:
def max_heap_size(nums, k):
    heap, biggest = [], 0
    for i, x in enumerate(nums):
        heapq.heappush(heap, (-x, i))
        while heap[0][1] <= i - k:
            heapq.heappop(heap)
        biggest = max(biggest, len(heap))
    return biggest

n, k = 10_000, 3
print("Tăng dần : heap lớn nhất =", max_heap_size(list(range(n)), k), "phần tử (dù k chỉ là 3!)")
print("Giảm dần : heap lớn nhất =", max_heap_size(list(range(n, 0, -1)), k), "phần tử")

Tăng dần : heap lớn nhất = 10000 phần tử (dù k chỉ là 3!)
Giảm dần : heap lớn nhất = 3 phần tử


## 5. Cách 3 — Chia khối kích thước `k` + prefix max / suffix max

Đây là một ý tưởng sáng tạo (trong xử lý ảnh nó được biết tới với tên **thuật toán van Herk / Gil–Werman**). Không cần heap, không cần deque — chỉ cần mảng và vài vòng `for`.

### Bước 1: cắt mảng thành các khối, mỗi khối đúng `k` phần tử

Cắt từ trái sang: chỉ số `0..k-1` là khối 0, `k..2k-1` là khối 1, ... (khối cuối có thể ngắn hơn). Chỉ số `i` thuộc khối `i // k`.

Example 1, `k = 3`:

```text
chỉ số:   0  1  2 |  3  4  5 |  6  7
nums  :   1  3 -1 | -3  5  3 |  6  7
khối  :   0  0  0 |  1  1  1 |  2  2
```

### Bước 2: quan sát then chốt

Một cửa sổ có độ dài **đúng bằng** độ dài khối (`k`). Vì vậy nó chỉ có hai khả năng:

- **trùng khít một khối** (khi `s` chia hết cho `k`), hoặc
- **vắt qua đúng một vạch chia**: nó gồm **phần đuôi** của một khối và **phần đầu** của khối kế tiếp. Không thể chạm tới 3 khối, vì muốn phủ trọn một khối ở giữa thì cửa sổ phải dài hơn `k`.

Ví dụ cửa sổ `s = 1` là `[3, -1, -3]`:

```text
chỉ số:   0 [1  2 |  3] 4  5 |  6  7
             └──┘    ↑
             đuôi    đầu
             khối 0  khối 1
```

Vậy max cửa sổ = max( **max đuôi khối trái**, **max đầu khối phải** ). Nếu ta **tính sẵn** "max của mọi đầu khối" và "max của mọi đuôi khối", mỗi cửa sổ chỉ tốn **một phép `max` của hai số**.

### Kiến thức nền: prefix max và suffix max

Bạn có thể đã gặp **prefix sum** (tổng tiền tố): `prefix[i] = a[0] + ... + a[i]`, tính dần bằng `prefix[i] = prefix[i-1] + a[i]`. **Prefix max** y hệt, chỉ thay `+` bằng `max`:

```text
prefix_max[i] = max(a[0], ..., a[i]) = max(prefix_max[i-1], a[i])
```

**Suffix max** (max hậu tố) là bản chạy **từ phải sang trái**:

```text
suffix_max[i] = max(a[i], ..., a[n-1]) = max(suffix_max[i+1], a[i])
```

Ví dụ `a = [2, 1, 5, 3]`:

| i | 0 | 1 | 2 | 3 |
|---|---:|---:|---:|---:|
| `a[i]` | 2 | 1 | 5 | 3 |
| `prefix_max[i]` | 2 | 2 | 5 | 5 |
| `suffix_max[i]` | 5 | 5 | 5 | 3 |

**Trong bài này**, ta tính prefix/suffix max **riêng trong từng khối**: đến đầu một khối mới thì "bắt đầu lại".

- `prefix_max` chạy trái → phải, **reset** khi `i % k == 0` (đầu khối). Nhắc lại: `i % k` là số dư khi chia `i` cho `k`; với `k = 3` thì `i = 0, 3, 6` cho dư `0`.
- `suffix_max` chạy phải → trái, **reset** khi `i` là **cuối khối**: `(i + 1) % k == 0`, hoặc `i == n - 1` (cuối mảng — khối cuối có thể ngắn).

In [14]:
a = [2, 1, 5, 3]
prefix_max = [a[0]]
for x in a[1:]:
    prefix_max.append(max(prefix_max[-1], x))
suffix_max = [a[-1]]
for x in reversed(a[:-1]):
    suffix_max.append(max(suffix_max[-1], x))
suffix_max.reverse()                     # đã xây từ phải sang trái nên đảo lại
print("a          =", a)
print("prefix_max =", prefix_max)
print("suffix_max =", suffix_max)

k = 3
print("\ni     :", list(range(8)))
print("i % k :", [i % k for i in range(8)], " <- 0 là đầu khối")
print("i // k:", [i // k for i in range(8)], " <- số thứ tự khối")

a          = [2, 1, 5, 3]
prefix_max = [2, 2, 5, 5]
suffix_max = [5, 5, 5, 3]

i     : [0, 1, 2, 3, 4, 5, 6, 7]
i % k : [0, 1, 2, 0, 1, 2, 0, 1]  <- 0 là đầu khối
i // k: [0, 0, 0, 1, 1, 1, 2, 2]  <- số thứ tự khối


### Bảng prefix/suffix max theo khối cho example 1

| i | `nums[i]` | khối `i // k` | `prefix_max[i]` (đầu khối → i) | `suffix_max[i]` (i → cuối khối) |
|---:|---:|---:|---:|---:|
| 0 | 1 | 0 | 1 | 3 |
| 1 | 3 | 0 | 3 | 3 |
| 2 | -1 | 0 | 3 | -1 |
| 3 | -3 | 1 | -3 | 5 |
| 4 | 5 | 1 | 5 | 5 |
| 5 | 3 | 1 | 5 | 3 |
| 6 | 6 | 2 | 6 | 7 |
| 7 | 7 | 2 | 7 | 7 |

Đọc vài ô cho quen:

- `prefix_max[4] = 5`: khối 1 bắt đầu ở chỉ số 3, nên đây là `max(nums[3], nums[4]) = max(-3, 5)`. Chú ý **không** tính `nums[0..2]` vì chúng thuộc khối khác.
- `suffix_max[3] = 5`: `max(nums[3], nums[4], nums[5]) = max(-3, 5, 3)`.
- `suffix_max[6] = 7`: khối cuối chỉ có 2 phần tử `6, 7`, cuối khối là cuối mảng (`i == n - 1`).

In [15]:
def build_block_max(nums, k):
    n = len(nums)
    prefix_max = [0] * n
    suffix_max = [0] * n
    for i in range(n):                            # trái -> phải
        if i % k == 0:                            # đầu khối: bắt đầu lại
            prefix_max[i] = nums[i]
        else:
            prefix_max[i] = max(prefix_max[i - 1], nums[i])
    for i in range(n - 1, -1, -1):                # phải -> trái
        if i == n - 1 or (i + 1) % k == 0:        # cuối khối (hoặc cuối mảng): bắt đầu lại
            suffix_max[i] = nums[i]
        else:
            suffix_max[i] = max(suffix_max[i + 1], nums[i])
    return prefix_max, suffix_max

pre, suf = build_block_max(nums1, k1)
print("nums      :", nums1)
print("prefix_max:", pre)
print("suffix_max:", suf)

nums      : [1, 3, -1, -3, 5, 3, 6, 7]
prefix_max: [1, 3, 3, -3, 5, 5, 6, 7]
suffix_max: [3, 3, -1, 5, 5, 3, 7, 7]


### Công thức ghép: `ans[s] = max(suffix_max[s], prefix_max[s + k - 1])`

Cửa sổ bắt đầu ở `s`, kết thúc ở `e = s + k - 1`.

- **Phần trái**: từ `s` tới **cuối khối chứa `s`** → max của nó chính là `suffix_max[s]`.
- **Phần phải**: từ **đầu khối chứa `e`** tới `e` → max của nó chính là `prefix_max[e]`.

Hai phần hợp lại **đúng bằng** cửa sổ (đã lập luận ở trên: cửa sổ chạm tối đa 2 khối). Khi `s` là đầu khối, `s` và `e` cùng một khối, hai phần **trùng nhau** và đều bằng cả khối — công thức vẫn đúng vì `max(a, a) = a`.

Ví dụ số với example 1:

- `s = 1`: phần trái `[3, -1]` → `suffix_max[1] = 3`; phần phải `[-3]` → `prefix_max[3] = -3`; đáp án `max(3, -3) = 3`. ✔
- `s = 4`: phần trái `[5, 3]` → `suffix_max[4] = 5`; phần phải `[6]` → `prefix_max[6] = 6`; đáp án `6`. ✔

In [16]:
def max_sliding_window_blocks(nums: List[int], k: int) -> List[int]:
    prefix_max, suffix_max = build_block_max(nums, k)
    return [max(suffix_max[s], prefix_max[s + k - 1]) for s in range(len(nums) - k + 1)]

print(max_sliding_window_blocks(nums1, k1))          # [3, 3, 5, 5, 6, 7]
print(max_sliding_window_blocks(nums2, k2))          # [1]
print(max_sliding_window_blocks(nums_ex2, k_ex2))    # [9, 7, 7, 4, 4, 4]

[3, 3, 5, 5, 6, 7]
[1]
[9, 7, 7, 4, 4, 4]


### Trace cách 3 trên example 1

| s | cửa sổ | phần trái (s → cuối khối) | `suffix_max[s]` | phần phải (đầu khối → s+k−1) | `prefix_max[s+k−1]` | max |
|---:|---|---|---:|---|---:|---:|
| 0 | `[1, 3, -1]` (trùng khít 1 khối) | `[1, 3, -1]` | 3 | `[1, 3, -1]` | 3 | **3** |
| 1 | `[3, -1, -3]` | `[3, -1]` | 3 | `[-3]` | -3 | **3** |
| 2 | `[-1, -3, 5]` | `[-1]` | -1 | `[-3, 5]` | 5 | **5** |
| 3 | `[-3, 5, 3]` (trùng khít 1 khối) | `[-3, 5, 3]` | 5 | `[-3, 5, 3]` | 5 | **5** |
| 4 | `[5, 3, 6]` | `[5, 3]` | 5 | `[6]` | 6 | **6** |
| 5 | `[3, 6, 7]` | `[3]` | 3 | `[6, 7]` | 7 | **7** |

Cell dưới in đúng bảng này bằng code, để bạn đổi input và tự quan sát.

In [17]:
def trace_blocks(nums, k):
    n = len(nums)
    pre, suf = build_block_max(nums, k)
    for s in range(n - k + 1):
        e = s + k - 1
        end_of_block_s = min((s // k) * k + k - 1, n - 1)   # cuối khối chứa s
        start_of_block_e = (e // k) * k                     # đầu khối chứa e
        left, right = nums[s:end_of_block_s + 1], nums[start_of_block_e:e + 1]
        print(f"s={s} {nums[s:e + 1]}: trái {left} -> suf={suf[s]:>2} | phải {right} -> pre={pre[e]:>2} | max={max(suf[s], pre[e])}")

trace_blocks(nums1, k1)

s=0 [1, 3, -1]: trái [1, 3, -1] -> suf= 3 | phải [1, 3, -1] -> pre= 3 | max=3
s=1 [3, -1, -3]: trái [3, -1] -> suf= 3 | phải [-3] -> pre=-3 | max=3
s=2 [-1, -3, 5]: trái [-1] -> suf=-1 | phải [-3, 5] -> pre= 5 | max=5
s=3 [-3, 5, 3]: trái [-3, 5, 3] -> suf= 5 | phải [-3, 5, 3] -> pre= 5 | max=5
s=4 [5, 3, 6]: trái [5, 3] -> suf= 5 | phải [6] -> pre= 6 | max=6
s=5 [3, 6, 7]: trái [3] -> suf= 3 | phải [6, 7] -> pre= 7 | max=7


### Độ phức tạp cách 3 và chỗ vướng

Gọi `n = len(nums)`.

- **Time:** `O(n)` — ba lượt duyệt độc lập (prefix, suffix, ghép), mỗi lượt `O(n)`, **không phụ thuộc `k`**.
- **Space:** `O(n)` cho hai mảng `prefix_max`, `suffix_max`.

Đã đạt `O(n)`! Cách này còn có ưu điểm hiếm: không có vòng `while` nào, mỗi phần tử được xử lý **đúng một số lần cố định** → rất dễ phân tích, và dễ song song hoá.

Nhưng nó có những hạn chế thực tế:

1. **Tốn `O(n)` bộ nhớ phụ** cho hai mảng, trong khi cửa sổ chỉ có `k` phần tử.
2. **Không chạy "online"** (dữ liệu đến từng phần tử một, như luồng giá cổ phiếu): `suffix_max` cần biết các phần tử **bên phải** — phải chờ đủ cả khối mới tính được.
3. **Chỉ hợp với cửa sổ độ dài cố định `k`**. Khi độ dài cửa sổ thay đổi (ví dụ LeetCode 1438 ở phần bài tập), cách chia khối không còn dùng được.

Ta muốn một cách `O(n)`, chạy online, bộ nhớ `O(k)`, và mở rộng được cho cửa sổ co giãn. Đó là **monotonic deque**.

## 6. Bước đột phá: phần tử "vô vọng" có thể **vứt bỏ vĩnh viễn**

Có một câu nói vui trong giới lập trình thi đấu:

> *"Nếu có người **trẻ hơn** bạn mà còn **giỏi hơn** bạn, thì bạn sẽ không bao giờ là người giỏi nhất nữa."*

Áp vào bài này: "trẻ hơn" = **vào cửa sổ sau** (chỉ số lớn hơn, nên sẽ ở lại lâu hơn), "giỏi hơn" = **giá trị lớn hơn hoặc bằng**.

**Khẳng định.** Nếu `j < t` và `nums[j] <= nums[t]`, thì kể từ lúc `t` vào cửa sổ, `j` **không bao giờ cần** làm đáp án nữa.

**Vì sao?** Xét một cửa sổ bất kỳ từ giờ trở đi (kết thúc tại `e >= t`) mà còn chứa `j`. Cửa sổ là một đoạn liền nhau, chứa `j` và kết thúc ở `e >= t > j` → nó chứa luôn `t`. Mà `nums[t] >= nums[j]` → max của cửa sổ đó luôn đạt được bằng `t`. Hơn nữa `j` sẽ rời cửa sổ **trước** `t`. Vậy `j` hoàn toàn vô dụng → **vứt luôn, không bao giờ cần nhìn lại**.

Ví dụ example 1: khi `3` (chỉ số 1) vào, `1` (chỉ số 0) trở thành vô vọng. Khi `5` (chỉ số 4) vào, cả `-1` và `-3` đều vô vọng.

### Ai còn lại? Định nghĩa "ứng viên"

Trong cửa sổ kết thúc tại `i`, gọi chỉ số `j` là **ứng viên** (còn hy vọng) nếu **mọi phần tử đứng sau `j`** trong cửa sổ đều **nhỏ hơn hẳn** `nums[j]`. Hai hệ quả quan trọng:

1. Đọc các ứng viên từ trái sang phải, **giá trị giảm dần** (giảm thật sự). Vì ứng viên đứng trước phải lớn hơn hẳn mọi thứ đứng sau nó, kể cả các ứng viên sau.
2. Ứng viên **đầu tiên** (trái nhất) chính là **max của cửa sổ**. (Phần tử max nằm ở vị trí xa nhất bên phải trong số các phần tử đạt max luôn là ứng viên, và nó lớn nhất trong các ứng viên.)

Cell dưới tìm ứng viên bằng brute force (đúng theo định nghĩa) để ta nhìn tận mắt hai hệ quả trên. Hàm `all(điều_kiện for ...)` trả `True` khi **mọi** phần tử thoả điều kiện (và trả `True` nếu không có phần tử nào — phần tử cuối cửa sổ luôn là ứng viên).

In [18]:
def candidates_bruteforce(nums, k, i):
    # các chỉ số còn hy vọng trong cửa sổ kết thúc tại i
    start = max(0, i - k + 1)
    result = []
    for j in range(start, i + 1):
        if all(nums[t] < nums[j] for t in range(j + 1, i + 1)):
            result.append(j)
    return result

for nums, k in [(nums1, k1), (nums_ex2, k_ex2)]:
    print("nums =", nums)
    for i in range(k - 1, len(nums)):
        c = candidates_bruteforce(nums, k, i)
        window = nums[i - k + 1:i + 1]
        print(f"  cửa sổ {str(window):<13} ứng viên (chỉ số) {str(c):<10} giá trị {str([nums[j] for j in c]):<12} max = {max(window)}")

nums = [1, 3, -1, -3, 5, 3, 6, 7]
  cửa sổ [1, 3, -1]    ứng viên (chỉ số) [1, 2]     giá trị [3, -1]      max = 3
  cửa sổ [3, -1, -3]   ứng viên (chỉ số) [1, 2, 3]  giá trị [3, -1, -3]  max = 3
  cửa sổ [-1, -3, 5]   ứng viên (chỉ số) [4]        giá trị [5]          max = 5
  cửa sổ [-3, 5, 3]    ứng viên (chỉ số) [4, 5]     giá trị [5, 3]       max = 5
  cửa sổ [5, 3, 6]     ứng viên (chỉ số) [6]        giá trị [6]          max = 6
  cửa sổ [3, 6, 7]     ứng viên (chỉ số) [7]        giá trị [7]          max = 7
nums = [9, 5, 7, 3, 1, 4, 4, 2]
  cửa sổ [9, 5, 7]     ứng viên (chỉ số) [0, 2]     giá trị [9, 7]       max = 9
  cửa sổ [5, 7, 3]     ứng viên (chỉ số) [2, 3]     giá trị [7, 3]       max = 7
  cửa sổ [7, 3, 1]     ứng viên (chỉ số) [2, 3, 4]  giá trị [7, 3, 1]    max = 7
  cửa sổ [3, 1, 4]     ứng viên (chỉ số) [5]        giá trị [4]          max = 4
  cửa sổ [1, 4, 4]     ứng viên (chỉ số) [6]        giá trị [4]          max = 4
  cửa sổ [4, 4, 2]     ứng viên (chỉ số) [6

### Nhìn vào output: danh sách ứng viên thay đổi **chỉ ở hai đầu**

So sánh danh sách ứng viên giữa hai cửa sổ liên tiếp (ví dụ thứ hai):

- `[7, 3]` → `[7, 3, 1]`: phần tử mới `1` được **nối vào cuối** (nó nhỏ hơn mọi ứng viên cũ nên không đuổi ai).
- `[7, 3, 1]` → `[4]`: `7` **hết hạn ở đầu**; `4` vào và **đuổi từ cuối** những ứng viên `<= 4` là `1` rồi `3`.
- `[4]` → `[4]`: số `4` mới vào đuổi số `4` cũ (bằng nhau → cũ vô vọng), rồi tự nằm vào.

Tất cả thao tác đều ở **hai đầu** của danh sách:

- **đầu trái**: bỏ ứng viên đã hết hạn (và đọc max);
- **đầu phải**: bỏ các ứng viên `<=` phần tử mới, rồi thêm phần tử mới.

Ta cần một cấu trúc thêm/bớt **ở cả hai đầu** trong `O(1)`. Đó là **deque**.

## 7. Kiến thức nền: queue, stack và `collections.deque`

- **Stack** (ngăn xếp): vào sau ra trước (LIFO — *Last In, First Out*), như chồng đĩa — chỉ thao tác ở **đỉnh**. List của Python làm stack rất tốt: `append` / `pop` ở cuối đều `O(1)`.
- **Queue** (hàng đợi): vào trước ra trước (FIFO — *First In, First Out*), như xếp hàng mua vé — vào ở **cuối**, ra ở **đầu**.
- **Deque** (*double-ended queue*, đọc là "đéc"): hàng đợi **hai đầu** — thêm/bớt được ở **cả đầu lẫn cuối**, tất cả đều `O(1)`.

Python có sẵn `deque` trong module `collections`:

| Lệnh | Ý nghĩa | Thời gian |
|---|---|---|
| `dq = deque()` | tạo deque rỗng | `O(1)` |
| `dq.append(x)` | thêm `x` vào **cuối** (bên phải) | `O(1)` |
| `dq.pop()` | lấy ra và xoá phần tử **cuối** | `O(1)` |
| `dq.appendleft(x)` | thêm `x` vào **đầu** (bên trái) | `O(1)` |
| `dq.popleft()` | lấy ra và xoá phần tử **đầu** | `O(1)` |
| `dq[0]`, `dq[-1]` | xem phần tử đầu / cuối | `O(1)` |
| `len(dq)`, `if dq:` | số phần tử / còn phần tử không | `O(1)` |

**Vì sao không dùng list?** `lst.pop(0)` (xoá phần tử đầu list) phải **dời tất cả** phần tử còn lại sang trái một ô → `O(m)` với `m` là độ dài list. Deque được cài đặt khác (một chuỗi các khối liên kết với nhau) nên bỏ phần tử đầu chỉ tốn `O(1)`.

Lưu ý nhỏ: `if dq:` là `True` khi deque **còn phần tử**, `False` khi rỗng. Ta sẽ viết `while dq and ...` để không bao giờ đọc `dq[-1]` của deque rỗng (sẽ báo lỗi `IndexError`).

In [19]:
from collections import deque

dq = deque()
dq.append(1); dq.append(2); dq.append(3)
print("sau 3 lần append      :", dq)
dq.appendleft(0)
print("appendleft(0)         :", dq)
print("đầu dq[0] =", dq[0], "| cuối dq[-1] =", dq[-1], "| len =", len(dq))
print("popleft() lấy ra      :", dq.popleft(), "->", dq)
print("pop() lấy ra          :", dq.pop(), "->", dq)
print("list(dq)              :", list(dq))
empty = deque()
print("deque rỗng là False?  :", not empty)

sau 3 lần append      : deque([1, 2, 3])
appendleft(0)         : deque([0, 1, 2, 3])
đầu dq[0] = 0 | cuối dq[-1] = 3 | len = 4
popleft() lấy ra      : 0 -> deque([1, 2, 3])
pop() lấy ra          : 3 -> deque([1, 2])
list(dq)              : [1, 2]
deque rỗng là False?  : True


In [20]:
import time

m = 50_000
lst = list(range(m))
t = time.perf_counter()
while lst:
    lst.pop(0)                  # mỗi lần dời cả list sang trái
t_list = time.perf_counter() - t

dq = deque(range(m))
t = time.perf_counter()
while dq:
    dq.popleft()                # O(1)
t_deque = time.perf_counter() - t

print(f"list.pop(0)    x {m}: {t_list:.4f} giây")
print(f"deque.popleft  x {m}: {t_deque:.4f} giây")
print("deque nhanh hơn nhiều lần:", t_list > t_deque)

list.pop(0)    x 50000: 0.1976 giây
deque.popleft  x 50000: 0.0019 giây
deque nhanh hơn nhiều lần: True


## 8. Cách 4 — Monotonic deque (cách chính)

Ta dùng một deque `dq` để lưu **chỉ số** của các ứng viên, theo đúng thứ tự trong mảng. Gọi là **monotonic** (đơn điệu) vì nó luôn giữ tính chất:

```text
chỉ số   : tăng dần từ đầu -> cuối
giá trị  : GIẢM THẬT SỰ từ đầu -> cuối    nums[dq[0]] > nums[dq[1]] > ...
đầu dq[0]: luôn là chỉ số của max cửa sổ hiện tại
```

Với mỗi `i` (phần tử `x = nums[i]` đi vào), làm **4 bước**:

1. **Đầu — bỏ chỉ số hết hạn:** nếu `dq[0] <= i - k` thì `dq.popleft()`.
2. **Đuôi — bỏ phần tử vô vọng:** `while dq and nums[dq[-1]] <= x: dq.pop()` (những ứng viên `<= x` vừa bị `x` "trẻ hơn mà giỏi hơn" đánh bại).
3. **Thêm mới:** `dq.append(i)` (`x` luôn là ứng viên vì nó đứng cuối cửa sổ).
4. **Ghi max:** nếu `i >= k - 1` thì thêm `nums[dq[0]]` vào kết quả.

Câu nhớ nhanh: **"Đầu hết hạn — Đuôi vô vọng — Thêm mới — Ghi đầu."**

### Ba câu hỏi người mới hay thắc mắc

**(a) Vì sao lưu chỉ số mà không lưu giá trị?** Bước 1 cần biết phần tử ở đầu **nằm ở đâu** để so với `i - k`. Giá trị không cho biết vị trí, nhất là khi có số trùng nhau (xem lỗi 4 ở mục "Lỗi người mới hay gặp"). Có chỉ số rồi thì giá trị lấy lại dễ dàng bằng `nums[j]`.

**(b) Vì sao bước 1 chỉ cần `if`, còn bước 2 phải `while`?** Trước bước 1, deque chỉ chứa chỉ số trong cửa sổ trước, tức `i-k .. i-1`. Khi trượt, **chỉ đúng một chỉ số** là `i - k` rời đi, và nếu nó còn trong deque thì nó phải nằm ở đầu (chỉ số tăng dần). Nên bỏ tối đa một phần tử → `if` là đủ (viết `while` cũng đúng, chỉ là thừa). Ngược lại, ở bước 2 một phần tử lớn đi vào có thể đuổi **nhiều** ứng viên cùng lúc → bắt buộc `while`.

**(c) Vì sao so sánh `<=` mà không phải `<`?** Với `<=`, phần tử cũ **bằng** phần tử mới cũng bị bỏ — hợp lý vì nó vô vọng (bằng giá trị nhưng rời đi sớm hơn). Nhờ đó giá trị trong deque **giảm thật sự**. Dùng `<` cũng cho kết quả đúng (deque chỉ giữ thêm các bản trùng, "không tăng" thay vì "giảm thật sự"), nhưng `<=` gọn hơn.

In [21]:
def max_sliding_window_deque(nums: List[int], k: int) -> List[int]:
    dq = deque()                                # chỉ số ứng viên, giá trị giảm dần
    result = []
    for i, x in enumerate(nums):
        if dq and dq[0] <= i - k:               # 1. đầu: bỏ chỉ số đã rời cửa sổ
            dq.popleft()
        while dq and nums[dq[-1]] <= x:         # 2. đuôi: bỏ phần tử vô vọng
            dq.pop()
        dq.append(i)                            # 3. thêm mới
        if i >= k - 1:                          # 4. đủ k phần tử: đầu deque là max
            result.append(nums[dq[0]])
    return result

print(max_sliding_window_deque(nums1, k1))          # [3, 3, 5, 5, 6, 7]
print(max_sliding_window_deque(nums2, k2))          # [1]
print(max_sliding_window_deque(nums_ex2, k_ex2))    # [9, 7, 7, 4, 4, 4]

[3, 3, 5, 5, 6, 7]
[1]
[9, 7, 7, 4, 4, 4]


### Trace cách 4 trên example 1 `[1,3,-1,-3,5,3,6,7]`, `k = 3`

Ý nghĩa các cột: "cửa sổ" là khoảng chỉ số của cửa sổ kết thúc tại `i`; cột `i − k` là ngưỡng hết hạn (chỉ số `<= i − k` đã rời đi); cột ① là bước 1 (đầu), cột ② là bước 2 (đuôi, ghi `chỉ số(giá trị)` theo thứ tự bị pop); deque được ghi **sau** bước 3.

| i | x | cửa sổ | i − k | ① đầu: hết hạn? | ② đuôi: vô vọng (≤ x) | deque (chỉ số) | giá trị tương ứng | ④ max |
|---:|---:|---|---:|---|---|---|---|---:|
| 0 | 1 | `0..0` (chưa đủ k) | -3 | — | — | `[0]` | `[1]` | — |
| 1 | 3 | `0..1` (chưa đủ k) | -2 | — | bỏ `0`(1) | `[1]` | `[3]` | — |
| 2 | -1 | `0..2` | -1 | — | — | `[1, 2]` | `[3, -1]` | **3** |
| 3 | -3 | `1..3` | 0 | — | — | `[1, 2, 3]` | `[3, -1, -3]` | **3** |
| 4 | 5 | `2..4` | 1 | **bỏ `1`** (giá trị 3) | bỏ `3`(-3), `2`(-1) | `[4]` | `[5]` | **5** |
| 5 | 3 | `3..5` | 2 | — | — | `[4, 5]` | `[5, 3]` | **5** |
| 6 | 6 | `4..6` | 3 | — | bỏ `5`(3), `4`(5) | `[6]` | `[6]` | **6** |
| 7 | 7 | `5..7` | 4 | — | bỏ `6`(6) | `[7]` | `[7]` | **7** |

Những dòng đáng chú ý:

- `i = 1`: `3` vào, đuổi `1` (chỉ số 0) — "trẻ hơn mà giỏi hơn".
- `i = 3`: `-3` nhỏ hơn mọi ứng viên nên chỉ được nối vào cuối. Deque `[1, 2, 3]` có giá trị `[3, -1, -3]` — giảm dần, đầu là max `3`.
- `i = 4`: **cả hai nhánh cùng chạy**. Bước 1: chỉ số 1 (giá trị 3) hết hạn (`1 <= 4 - 3`) → `popleft`. Bước 2: `5` đuổi `-3` rồi `-1`. Deque chỉ còn `[4]`.
- `i = 6`: `6` đuổi một lúc **hai** ứng viên `3` và `5`.

In [22]:
def trace_deque(nums, k):
    dq = deque()
    result = []
    for i, x in enumerate(nums):
        front = "-"
        if dq and dq[0] <= i - k:
            j = dq.popleft()
            front = f"bỏ {j}({nums[j]})"
        back = []
        while dq and nums[dq[-1]] <= x:
            j = dq.pop()
            back.append(f"{j}({nums[j]})")
        dq.append(i)
        line = (f"i={i} x={x:>2} | đầu: {front:<9} | đuôi: {', '.join(back) if back else '-':<14}"
                f" | dq={str(list(dq)):<10} giá trị={str([nums[j] for j in dq]):<12}")
        if i >= k - 1:
            result.append(nums[dq[0]])
            line += f" -> max={nums[dq[0]]}"
        print(line)
    return result

print("Kết quả:", trace_deque(nums1, k1))

i=0 x= 1 | đầu: -         | đuôi: -              | dq=[0]        giá trị=[1]         
i=1 x= 3 | đầu: -         | đuôi: 0(1)           | dq=[1]        giá trị=[3]         
i=2 x=-1 | đầu: -         | đuôi: -              | dq=[1, 2]     giá trị=[3, -1]      -> max=3
i=3 x=-3 | đầu: -         | đuôi: -              | dq=[1, 2, 3]  giá trị=[3, -1, -3]  -> max=3
i=4 x= 5 | đầu: bỏ 1(3)   | đuôi: 3(-3), 2(-1)   | dq=[4]        giá trị=[5]          -> max=5
i=5 x= 3 | đầu: -         | đuôi: -              | dq=[4, 5]     giá trị=[5, 3]       -> max=5
i=6 x= 6 | đầu: -         | đuôi: 5(3), 4(5)     | dq=[6]        giá trị=[6]          -> max=6
i=7 x= 7 | đầu: -         | đuôi: 6(6)           | dq=[7]        giá trị=[7]          -> max=7
Kết quả: [3, 3, 5, 5, 6, 7]


### Trace thứ hai: số lớn nhất **hết hạn** và số **trùng nhau** — `[9,5,7,3,1,4,4,2]`, `k = 3`

| i | x | cửa sổ | i − k | ① đầu: hết hạn? | ② đuôi: vô vọng (≤ x) | deque (chỉ số) | giá trị tương ứng | ④ max |
|---:|---:|---|---:|---|---|---|---|---:|
| 0 | 9 | `0..0` (chưa đủ k) | -3 | — | — | `[0]` | `[9]` | — |
| 1 | 5 | `0..1` (chưa đủ k) | -2 | — | — | `[0, 1]` | `[9, 5]` | — |
| 2 | 7 | `0..2` | -1 | — | bỏ `1`(5) | `[0, 2]` | `[9, 7]` | **9** |
| 3 | 3 | `1..3` | 0 | **bỏ `0`** (giá trị 9) | — | `[2, 3]` | `[7, 3]` | **7** |
| 4 | 1 | `2..4` | 1 | — | — | `[2, 3, 4]` | `[7, 3, 1]` | **7** |
| 5 | 4 | `3..5` | 2 | **bỏ `2`** (giá trị 7) | bỏ `4`(1), `3`(3) | `[5]` | `[4]` | **4** |
| 6 | 4 | `4..6` | 3 | — | bỏ `5`(4) | `[6]` | `[4]` | **4** |
| 7 | 2 | `5..7` | 4 | — | — | `[6, 7]` | `[4, 2]` | **4** |

- `i = 2`: `7` đuổi `5` nhưng **không** đuổi được `9` (`9 > 7`) → deque `[0, 2]` có giá trị `[9, 7]`. Số `7` được giữ lại làm "người kế vị".
- `i = 3`: `9` hết hạn → `popleft`. Max mới là `7` có ngay ở đầu deque, **không phải quét lại** — đây chính là thứ cách 1b thiếu!
- `i = 5`: `7` hết hạn ở đầu, đồng thời `4` đuổi `1` và `3` ở đuôi.
- `i = 6`: số `4` mới đuổi số `4` cũ (vì `<=`). Kết quả vẫn đúng `4`, và deque gọn hơn.

In [23]:
print("Kết quả:", trace_deque(nums_ex2, k_ex2))

i=0 x= 9 | đầu: -         | đuôi: -              | dq=[0]        giá trị=[9]         
i=1 x= 5 | đầu: -         | đuôi: -              | dq=[0, 1]     giá trị=[9, 5]      
i=2 x= 7 | đầu: -         | đuôi: 1(5)           | dq=[0, 2]     giá trị=[9, 7]       -> max=9
i=3 x= 3 | đầu: bỏ 0(9)   | đuôi: -              | dq=[2, 3]     giá trị=[7, 3]       -> max=7
i=4 x= 1 | đầu: -         | đuôi: -              | dq=[2, 3, 4]  giá trị=[7, 3, 1]    -> max=7
i=5 x= 4 | đầu: bỏ 2(7)   | đuôi: 4(1), 3(3)     | dq=[5]        giá trị=[4]          -> max=4
i=6 x= 4 | đầu: -         | đuôi: 5(4)           | dq=[6]        giá trị=[4]          -> max=4
i=7 x= 2 | đầu: -         | đuôi: -              | dq=[6, 7]     giá trị=[4, 2]       -> max=4
Kết quả: [9, 7, 7, 4, 4, 4]


### Vì sao đúng? (invariant bằng lời)

**Invariant** (bất biến) là điều luôn đúng sau mỗi vòng lặp. Sau khi xử lý xong chỉ số `i` (sau bước 3), deque thoả cả ba điều:

- **(I1) Đúng cửa sổ:** mọi chỉ số trong deque nằm trong cửa sổ `i-k+1 .. i`, và tăng dần từ đầu tới cuối.
- **(I2) Giảm thật sự:** `nums[dq[0]] > nums[dq[1]] > ... > nums[dq[-1]]`, và `dq[-1] = i`.
- **(I3) Không bỏ sót:** mọi chỉ số `j` trong cửa sổ mà **không** có trong deque đều bị một chỉ số **đứng sau nó, có giá trị `>=`** "chắn" — tức `j` vô vọng.

Vì sao giữ được?

- (I1): bước 1 bỏ chỉ số duy nhất vừa rời đi (`i - k`); bước 3 thêm `i` vào cuối, mà `i` lớn hơn mọi chỉ số cũ.
- (I2): bước 2 bỏ mọi phần tử cuối `<= x` cho tới khi gặp phần tử `> x` (hoặc deque rỗng); vậy khi `x` được nối vào, nó nhỏ hơn hẳn phần tử đứng trước → dãy vẫn giảm thật sự.
- (I3): phần tử bị bỏ ở bước 2 bị chính `i` chắn (`i` đứng sau, `nums[i] >= ` nó). Phần tử bị chắn từ trước thì người chắn nó (hoặc người chắn người chắn nó, ...) vẫn còn trong cửa sổ, vì người chắn đứng sau nên rời cửa sổ muộn hơn.

**Kết luận:** max của cửa sổ là một phần tử không bị ai chắn (lấy phần tử đạt max đứng xa nhất bên phải) → nó có trong deque theo (I3); deque giảm dần theo (I2) → nó nằm ở **đầu**. Vậy `nums[dq[0]]` đúng là max. Thực ra deque **chính là** danh sách ứng viên ta đã định nghĩa ở mục 6.

Cell dưới kiểm tra cả ba điều bằng `assert` tại **mọi bước** trên 300 mảng ngẫu nhiên (giá trị nhỏ để có nhiều số trùng).

In [24]:
def check_invariants(nums, k):
    dq = deque()
    for i, x in enumerate(nums):
        if dq and dq[0] <= i - k:
            dq.popleft()
        while dq and nums[dq[-1]] <= x:
            dq.pop()
        dq.append(i)
        lo = max(0, i - k + 1)
        assert all(lo <= j <= i for j in dq)                                  # (I1)
        assert all(nums[dq[t]] > nums[dq[t + 1]] for t in range(len(dq) - 1))   # (I2)
        assert list(dq) == candidates_bruteforce(nums, k, i)                  # (I3): đúng bằng tập ứng viên
        assert nums[dq[0]] == max(nums[lo:i + 1])                              # đầu deque là max

random.seed(239)
for _ in range(300):
    arr = [random.randint(-3, 3) for _ in range(random.randint(1, 12))]
    check_invariants(arr, random.randint(1, len(arr)))
print("Cả ba invariant đúng ở mọi bước trên 300 mảng ngẫu nhiên: PASS")

Cả ba invariant đúng ở mọi bước trên 300 mảng ngẫu nhiên: PASS


### Phân tích **amortized** `O(n)` — vì sao `for` lồng `while` mà không phải `O(n·k)`?

Nhìn code, người mới thường nghĩ: *"vòng `for` chạy `n` lần, bên trong có `while` chạy tới `k` lần → `O(n·k)`"*. Sai! Ta phải **đếm tổng** thay vì nhân "tối đa mỗi vòng".

Lập luận đếm:

- Mỗi chỉ số `0..n-1` được `append` vào deque **đúng 1 lần** (bước 3).
- Một chỉ số đã bị `pop` hay `popleft` thì **không bao giờ quay lại**. Nên mỗi chỉ số bị lấy ra **tối đa 1 lần**.
- Vậy **tổng** số lần chạy thân `while` (cộng lại qua tất cả các vòng `for`) ≤ `n`, và tổng số lần `popleft` ≤ `n`.

→ Tổng công việc ≤ `n` (append) + `n` (pop) + `n` (popleft) + `n` (phần còn lại của mỗi vòng) = `O(n)`.

Đây gọi là phân tích **amortized** (khấu hao, hay "chia đều"): một bước riêng lẻ có thể đắt (ở example 1, bước `i = 4` làm 3 lần pop), nhưng các bước đắt **"trả bằng"** những lần append rẻ trước đó. Hình dung: mỗi người vào cửa hàng **một lần** và ra **một lần**. Có lúc một nhóm 3 người ra cùng lúc, nhưng tổng số lượt ra **không bao giờ vượt** tổng số người đã vào.

Kiểm chứng trên example 1: 8 lần append; số lần pop: `i=1` một lần, `i=4` ba lần (1 popleft + 2 pop), `i=6` hai lần, `i=7` một lần → tổng 7 ≤ 8.

In [25]:
def count_operations(nums, k):
    dq = deque()
    pushes = back_pops = front_pops = worst_step = 0
    for i, x in enumerate(nums):
        step = 0
        if dq and dq[0] <= i - k:
            dq.popleft(); front_pops += 1; step += 1
        while dq and nums[dq[-1]] <= x:
            dq.pop(); back_pops += 1; step += 1
        dq.append(i); pushes += 1
        worst_step = max(worst_step, step)
    return pushes, back_pops, front_pops, worst_step

print("example 1:", count_operations(nums1, k1), "(push, pop đuôi, popleft, nhiều nhất trong 1 bước)")

n, k = 10**5, 1000
random.seed(0)
tests = {
    "ngẫu nhiên": [random.randint(-10**4, 10**4) for _ in range(n)],
    "tăng dần":   list(range(n)),
    "giảm dần":   list(range(n, 0, -1)),
    "hằng số":    [7] * n,
    "răng cưa giảm": [k - 1 - i % k for i in range(n)],   # 999, 998, ..., 0, 999, 998, ..., 0, ...
}
print(f"\nn = {n:,}, k = {k}")
for name, arr in tests.items():
    p, b, f, w = count_operations(arr, k)
    print(f"{name:<13}: push={p:,}  pop đuôi={b:,}  popleft={f:,}  tổng pop={b + f:,} <= n? {b + f <= n}  | 1 bước nhiều nhất: {w}")

example 1: (8, 6, 1, 3) (push, pop đuôi, popleft, nhiều nhất trong 1 bước)

n = 100,000, k = 1000
ngẫu nhiên   : push=100,000  pop đuôi=99,890  popleft=101  tổng pop=99,991 <= n? True  | 1 bước nhiều nhất: 14
tăng dần     : push=100,000  pop đuôi=99,999  popleft=0  tổng pop=99,999 <= n? True  | 1 bước nhiều nhất: 1
giảm dần     : push=100,000  pop đuôi=0  popleft=99,000  tổng pop=99,000 <= n? True  | 1 bước nhiều nhất: 1
hằng số      : push=100,000  pop đuôi=99,999  popleft=0  tổng pop=99,999 <= n? True  | 1 bước nhiều nhất: 1


răng cưa giảm: push=100,000  pop đuôi=98,901  popleft=99  tổng pop=99,000 <= n? True  | 1 bước nhiều nhất: 1000


Đọc output:

- Mảng **"răng cưa giảm"** (`999, 998, ..., 0, 999, 998, ..., 0, ...`): mỗi khi số `999` của chu kỳ mới đi vào, nó cho số `999` cũ hết hạn ở đầu rồi đuổi **cả chu kỳ trước** ở đuôi — **một bước** làm tới `1000` lần pop. Vậy mà **tổng** số pop vẫn không vượt `n`, vì những bước đắt như thế rất hiếm.
- Mảng **tăng dần**: bước nào cũng pop đúng 1 phần tử ở đuôi (số mới đuổi số liền trước) — tổng `n - 1`.
- Mảng **giảm dần**: không bao giờ pop đuôi, nhưng từ khi đủ cửa sổ thì bước nào cũng `popleft` — tổng `n - k`.
- Mảng **hằng số**: vì so `<=`, số mới luôn đuổi số cũ bằng nó — deque lúc nào cũng chỉ có 1 phần tử.

### Độ phức tạp cách 4

Gọi `n = len(nums)`.

- **Time:** `O(n)` (amortized như trên) — **không phụ thuộc `k`**.
- **Space:** `O(k)` cho deque — theo (I1) mọi chỉ số trong deque nằm trong cửa sổ, nên deque có tối đa `k` phần tử. (List kết quả `n - k + 1` phần tử là bắt buộc phải trả về, thường không tính.)

So với cách 3: cùng `O(n)` nhưng chỉ tốn `O(k)` bộ nhớ, xử lý từng phần tử ngay khi nó tới (online), và — như bạn sẽ thấy ở bài tập 5 — mở rộng được cho cửa sổ co giãn.

## 9. Code nộp LeetCode

Bản submit dùng cách 4 (monotonic deque). Đúng signature `maxSlidingWindow(self, nums: List[int], k: int) -> List[int]`, không `print`. Trên LeetCode, `List` và `deque` đều đã được import sẵn, nhưng viết `from collections import deque` cũng không sao.

Cell dưới kèm bộ test: hai example của đề và các edge case (trường hợp biên) dễ sai:

- `k = 1` (mỗi phần tử là một cửa sổ) và `k = n` (chỉ một cửa sổ);
- mảng tăng dần, giảm dần (max luôn hết hạn), toàn số bằng nhau, toàn số âm;
- số trùng nhau nằm xen kẽ;
- giá trị biên `-10^4` và `10^4`.

Ngay sau bản chính là **hai bản nộp thay thế** (heap `O(n log n)` và chia khối `O(n)`) — cũng đều được Accepted. Khi nộp, đổi tên class của chúng về `Solution`; ở đây đặt tên khác để không đè lên bản chính.

In [26]:
from collections import deque
from typing import List


class Solution:
    def maxSlidingWindow(self, nums: List[int], k: int) -> List[int]:
        dq = deque()                      # chỉ số, giá trị giảm thật sự từ đầu -> cuối
        result = []
        for i, x in enumerate(nums):
            if dq and dq[0] <= i - k:     # đầu: hết hạn
                dq.popleft()
            while dq and nums[dq[-1]] <= x:   # đuôi: vô vọng
                dq.pop()
            dq.append(i)                  # thêm mới
            if i >= k - 1:                # ghi đầu
                result.append(nums[dq[0]])
        return result


def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<28} PASS  {actual}")

sol = Solution()
check("Example 1", sol.maxSlidingWindow([1, 3, -1, -3, 5, 3, 6, 7], 3), [3, 3, 5, 5, 6, 7])
check("Example 2", sol.maxSlidingWindow([1], 1), [1])
check("k = 1", sol.maxSlidingWindow([4, -2, 7], 1), [4, -2, 7])
check("k = n", sol.maxSlidingWindow([4, -2, 7, 1], 4), [7])
check("tăng dần", sol.maxSlidingWindow([1, 2, 3, 4, 5], 2), [2, 3, 4, 5])
check("giảm dần (max hết hạn)", sol.maxSlidingWindow([5, 4, 3, 2, 1], 2), [5, 4, 3, 2])
check("toàn số bằng nhau", sol.maxSlidingWindow([2, 2, 2, 2], 3), [2, 2])
check("toàn số âm", sol.maxSlidingWindow([-7, -8, -3, -9], 2), [-7, -3, -3])
check("trùng xen kẽ", sol.maxSlidingWindow([1, 3, 1, 3, 1], 2), [3, 3, 3, 3])
check("ví dụ thứ hai", sol.maxSlidingWindow([9, 5, 7, 3, 1, 4, 4, 2], 3), [9, 7, 7, 4, 4, 4])
check("giá trị biên", sol.maxSlidingWindow([-10**4, 10**4, -10**4], 2), [10**4, 10**4])

Example 1                    PASS  [3, 3, 5, 5, 6, 7]
Example 2                    PASS  [1]
k = 1                        PASS  [4, -2, 7]
k = n                        PASS  [7]
tăng dần                     PASS  [2, 3, 4, 5]
giảm dần (max hết hạn)       PASS  [5, 4, 3, 2]
toàn số bằng nhau            PASS  [2, 2]
toàn số âm                   PASS  [-7, -3, -3]
trùng xen kẽ                 PASS  [3, 3, 3, 3]
ví dụ thứ hai                PASS  [9, 7, 7, 4, 4, 4]
giá trị biên                 PASS  [10000, 10000]


In [27]:
# Bản nộp thay thế: khi nộp lên LeetCode, đổi tên class về Solution
class SolutionHeap:
    def maxSlidingWindow(self, nums: List[int], k: int) -> List[int]:
        heap, result = [], []
        for i, x in enumerate(nums):
            heapq.heappush(heap, (-x, i))
            while heap[0][1] <= i - k:
                heapq.heappop(heap)
            if i >= k - 1:
                result.append(-heap[0][0])
        return result


class SolutionBlocks:
    def maxSlidingWindow(self, nums: List[int], k: int) -> List[int]:
        n = len(nums)
        pre, suf = nums[:], nums[:]               # copy để có sẵn giá trị ở đầu/cuối khối
        for i in range(1, n):
            if i % k != 0:
                pre[i] = max(pre[i - 1], nums[i])
        for i in range(n - 2, -1, -1):
            if (i + 1) % k != 0:
                suf[i] = max(suf[i + 1], nums[i])
        return [max(suf[s], pre[s + k - 1]) for s in range(n - k + 1)]


for cls in [SolutionHeap, SolutionBlocks]:
    s = cls()
    assert s.maxSlidingWindow([1, 3, -1, -3, 5, 3, 6, 7], 3) == [3, 3, 5, 5, 6, 7]
    assert s.maxSlidingWindow([1], 1) == [1]
    assert s.maxSlidingWindow([9, 5, 7, 3, 1, 4, 4, 2], 3) == [9, 7, 7, 4, 4, 4]
    assert s.maxSlidingWindow([5, 4, 3, 2, 1], 2) == [5, 4, 3, 2]
    print(cls.__name__, "PASS")

SolutionHeap PASS
SolutionBlocks PASS


### Thử với input lớn nhất theo constraints

`n = 10^5`, giá trị trong `[-10^4, 10^4]`, và `k = 50 000` (trường hợp tệ nhất của brute force). Ba cách nhanh chạy trong vài phần trăm giây. Brute force với cỡ này cần khoảng `2.5 · 10^9` bước nên ta **không** chạy nó; thay vào đó chạy nó trên input nhỏ hơn 10 lần (`n = 10^4`, `k = 5000`, tức khoảng `2.5 · 10^7` bước) để bạn ước lượng: input lớn gấp 10 lần thì brute force chậm đi khoảng **100 lần** (vì `n·k` tăng theo bình phương).

(Thời gian in ra phụ thuộc máy, mỗi lần chạy sẽ khác một chút; điều đáng xem là **tỉ lệ** giữa các cách.)

In [28]:
def timeit(fn, *args):
    t = time.perf_counter()
    out = fn(*args)
    return out, time.perf_counter() - t

random.seed(10**5)
big = [random.randint(-10**4, 10**4) for _ in range(10**5)]
k = 50_000
r_deque, t1 = timeit(Solution().maxSlidingWindow, big, k)
r_heap, t2 = timeit(max_sliding_window_heap, big, k)
r_block, t3 = timeit(max_sliding_window_blocks, big, k)
assert r_deque == r_heap == r_block
print(f"n = 10^5, k = {k}: deque {t1:.3f}s | heap {t2:.3f}s | chia khối {t3:.3f}s | cùng kết quả, {len(r_deque)} cửa sổ")

small = big[:10**4]
r_brute, t4 = timeit(max_sliding_window_bruteforce, small, 5000)
r_small, t5 = timeit(Solution().maxSlidingWindow, small, 5000)
assert r_brute == r_small
print(f"n = 10^4, k = 5000  : brute force {t4:.3f}s | deque {t5:.4f}s")
print(f"Ước lượng brute force ở n = 10^5, k = 50000: khoảng {t4 * 100:.0f}s -> TLE")

n = 10^5, k = 50000: deque 0.020s | heap 0.039s | chia khối 0.023s | cùng kết quả, 50001 cửa sổ


n = 10^4, k = 5000  : brute force 0.294s | deque 0.0019s
Ước lượng brute force ở n = 10^5, k = 50000: khoảng 29s -> TLE


### Kiểm thử đối chiếu chéo tất cả các cách

Ta chạy **cả 5 lời giải** (brute force, 1b, heap, chia khối, deque) cùng bản `Solution` trên hàng nghìn input ngẫu nhiên nhỏ và so với brute force — "trọng tài" chậm nhưng chắc chắn đúng. `random.seed(...)` cố định giúp mỗi lần chạy sinh ra **cùng** dãy input, nên nếu có lỗi bạn tái hiện được đúng ca đó.

Hai nhóm input: giá trị trong `[-3, 3]` (rất nhiều số trùng — hay làm lộ lỗi `<` / `<=`) và giá trị trong `[-10^4, 10^4]` (như đề).

In [29]:
random.seed(2026)
methods = {
    "rescan (1b)": lambda a, k: max_sliding_window_rescan(a, k)[0],
    "heap":        max_sliding_window_heap,
    "chia khối":   max_sliding_window_blocks,
    "deque":       max_sliding_window_deque,
    "Solution":    Solution().maxSlidingWindow,
    "SolutionHeap": SolutionHeap().maxSlidingWindow,
    "SolutionBlocks": SolutionBlocks().maxSlidingWindow,
}
total = 0
for lo, hi in [(-3, 3), (-10**4, 10**4)]:
    for _ in range(1500):
        arr = [random.randint(lo, hi) for _ in range(random.randint(1, 15))]
        k = random.randint(1, len(arr))
        expected = max_sliding_window_bruteforce(arr, k)
        for name, fn in methods.items():
            assert fn(arr, k) == expected, (name, arr, k)
        total += 1
print(f"{len(methods)} cách khớp brute force trên {total} input ngẫu nhiên: PASS")

7 cách khớp brute force trên 3000 input ngẫu nhiên: PASS


## 10. So sánh các cách

Gọi `n = len(nums)`, `k` là kích thước cửa sổ. Space không tính list kết quả.

| Cách | Ý tưởng chính | Time | Space | Chạy online? | Ghi chú |
|---|---|---|---|:---:|---|
| 1. Brute force | `max(nums[s:s+k])` cho từng cửa sổ | `O(n·k)` | `O(k)` | có | Dễ nhất, làm "trọng tài" để test; TLE khi `k ≈ n/2` |
| 1b. Giữ max, quét lại khi cần | 3 trường hợp `entering` / `leaving` | `O(n·k)` worst | `O(k)` | có | Nhanh với dữ liệu ngẫu nhiên, nhưng mảng giảm dần là worst case |
| 2. Max-heap + lazy deletion | heap `(-x, i)`, dọn đỉnh hết hạn | `O(n log n)` | `O(n)` | có | Ý tưởng "dọn lười" dùng lại được ở nhiều bài (vd 480) |
| 3. Chia khối + prefix/suffix max | `max(suffix[s], prefix[s+k-1])` | `O(n)` | `O(n)` | **không** | Không có `while`, dễ song song hoá; chỉ cho cửa sổ cố định |
| **4. Monotonic deque** | giữ chỉ số ứng viên, giá trị giảm dần | **`O(n)`** amortized | **`O(k)`** | có | **Cách chính**; mở rộng cho cửa sổ co giãn và DP |

Lời khuyên: trong phỏng vấn, hãy nói brute force trước (1 câu), nêu heap như bước trung gian, rồi trình bày monotonic deque kèm lập luận amortized. Cách chia khối là "điểm cộng" nếu người phỏng vấn hỏi *"có cách `O(n)` nào không dùng deque không?"*.

## 11. Lỗi người mới hay gặp

### Lỗi 1: Quên bỏ chỉ số hết hạn ở đầu deque

```python
for i, x in enumerate(nums):
    while dq and nums[dq[-1]] <= x:
        dq.pop()
    dq.append(i)                      # thiếu bước 1!
    if i >= k - 1:
        result.append(nums[dq[0]])
```

Max cũ (đã rời cửa sổ) vẫn nằm ở đầu nên bị trả lại mãi. Ví dụ `[9, 1, 1, 1], k = 2` cho `[9, 9, 9]` thay vì `[9, 1, 1]`. **Sửa:** thêm `if dq and dq[0] <= i - k: dq.popleft()`.

### Lỗi 2: Lệch một (off-by-one) ở điều kiện hết hạn

Viết `dq[0] < i - k` thay vì `dq[0] <= i - k` → chỉ số `i - k` (đã rời cửa sổ) vẫn được giữ, cửa sổ thực chất dài `k + 1`. Ví dụ `[5, 1, 1, 1], k = 2` cho `[5, 5, 1]` thay vì `[5, 1, 1]`. **Sửa:** nhớ cửa sổ kết thúc tại `i` là `i-k+1 .. i`, nên chỉ số **`<= i - k`** là hết hạn. Thử bằng số: `i = 2, k = 2` → cửa sổ `1..2` → chỉ số `0` hết hạn, và `0 <= 2 - 2` đúng.

### Lỗi 3: Dùng `if` thay cho `while` ở bước đuôi

```python
if dq and nums[dq[-1]] <= x:      # chỉ đuổi được MỘT phần tử
    dq.pop()
```

Phần tử lớn đi vào phải đuổi **mọi** ứng viên `<=` nó. Dùng `if` thì deque không còn giảm dần, đầu deque không còn là max. Ví dụ `[3, 1, 5], k = 3` cho `[3]` thay vì `[5]`. **Sửa:** `while`.

### Lỗi 4: Lưu **giá trị** vào deque và so `<=`

```python
if i >= k and dq and dq[0] == nums[i - k]:   # "phần tử rời đi có phải max không?"
    dq.popleft()
while dq and dq[-1] <= x:                     # so <= trên GIÁ TRỊ
    dq.pop()
dq.append(x)
```

Với số trùng, `<=` đã đuổi bản cũ và giữ bản mới; nhưng khi bản cũ rời cửa sổ, `dq[0] == nums[i-k]` vẫn đúng (cùng giá trị) nên **bản mới bị xoá nhầm**. Ví dụ `[3, 3, 1], k = 2` cho `[3, 1]` thay vì `[3, 3]`. **Sửa:** lưu **chỉ số** (cách chuẩn). Nếu nhất định lưu giá trị thì phải so `<` (giữ lại các bản trùng).

### Lỗi 5: Ghi kết quả khi cửa sổ chưa đủ `k` phần tử

Quên `if i >= k - 1:` → kết quả có `n` phần tử thay vì `n - k + 1`. Example 1 cho `[1, 3, 3, 3, 5, 5, 6, 7]`. **Sửa:** chỉ ghi khi `i >= k - 1`.

### Lỗi 6 (cách heap): dọn đỉnh bằng `if` thay cho `while`

Với lazy deletion, phần tử ngay dưới đỉnh có thể **cũng đã hết hạn từ lâu** (như `(-5, 1)` ở trace heap). `if` chỉ bỏ một đỉnh → đỉnh mới vẫn là rác. Ví dụ `[8, 9, 1, 1], k = 2` cho `[9, 9, 8]` thay vì `[9, 9, 1]`. **Sửa:** `while heap[0][1] <= i - k`. Một lỗi heap khác: quên đổi dấu khi đọc max (`heap[0][0]` thay vì `-heap[0][0]`) → ra số âm.

### Lỗi 7: Dùng `list.pop(0)` làm hàng đợi

Code vẫn **đúng** nhưng mỗi lần `pop(0)` tốn `O(k)` → có thể chậm hẳn (đã đo ở mục 7). **Sửa:** `collections.deque` với `popleft()`.

Cell dưới chạy các phiên bản sai để bạn thấy tận mắt output sai.

In [30]:
def wrong_no_expire(nums, k):                      # Lỗi 1
    dq, res = deque(), []
    for i, x in enumerate(nums):
        while dq and nums[dq[-1]] <= x:
            dq.pop()
        dq.append(i)
        if i >= k - 1:
            res.append(nums[dq[0]])
    return res

def wrong_off_by_one(nums, k):                     # Lỗi 2
    dq, res = deque(), []
    for i, x in enumerate(nums):
        if dq and dq[0] < i - k:                   # sai: phải là <=
            dq.popleft()
        while dq and nums[dq[-1]] <= x:
            dq.pop()
        dq.append(i)
        if i >= k - 1:
            res.append(nums[dq[0]])
    return res

def wrong_if_back(nums, k):                        # Lỗi 3
    dq, res = deque(), []
    for i, x in enumerate(nums):
        if dq and dq[0] <= i - k:
            dq.popleft()
        if dq and nums[dq[-1]] <= x:               # sai: phải là while
            dq.pop()
        dq.append(i)
        if i >= k - 1:
            res.append(nums[dq[0]])
    return res

def wrong_store_values(nums, k):                   # Lỗi 4
    dq, res = deque(), []
    for i, x in enumerate(nums):
        if i >= k and dq and dq[0] == nums[i - k]:
            dq.popleft()
        while dq and dq[-1] <= x:
            dq.pop()
        dq.append(x)
        if i >= k - 1:
            res.append(dq[0])
    return res

def wrong_record_early(nums, k):                   # Lỗi 5
    dq, res = deque(), []
    for i, x in enumerate(nums):
        if dq and dq[0] <= i - k:
            dq.popleft()
        while dq and nums[dq[-1]] <= x:
            dq.pop()
        dq.append(i)
        res.append(nums[dq[0]])                    # sai: thiếu if i >= k - 1
    return res

def wrong_heap_if(nums, k):                        # Lỗi 6
    heap, res = [], []
    for i, x in enumerate(nums):
        heapq.heappush(heap, (-x, i))
        if heap[0][1] <= i - k:                    # sai: phải là while
            heapq.heappop(heap)
        if i >= k - 1:
            res.append(-heap[0][0])
    return res

cases = [
    ("Lỗi 1 quên hết hạn", wrong_no_expire, [9, 1, 1, 1], 2),
    ("Lỗi 2 dùng <", wrong_off_by_one, [5, 1, 1, 1], 2),
    ("Lỗi 3 if ở đuôi", wrong_if_back, [3, 1, 5], 3),
    ("Lỗi 4 lưu giá trị", wrong_store_values, [3, 3, 1], 2),
    ("Lỗi 5 ghi sớm", wrong_record_early, [1, 3, -1, -3, 5, 3, 6, 7], 3),
    ("Lỗi 6 heap dùng if", wrong_heap_if, [8, 9, 1, 1], 2),
]
for name, fn, arr, k in cases:
    print(f"{name:<20} nums={str(arr):<27} k={k}: sai -> {str(fn(arr, k)):<26} đúng -> {Solution().maxSlidingWindow(arr, k)}")

Lỗi 1 quên hết hạn   nums=[9, 1, 1, 1]                k=2: sai -> [9, 9, 9]                  đúng -> [9, 1, 1]
Lỗi 2 dùng <         nums=[5, 1, 1, 1]                k=2: sai -> [5, 5, 1]                  đúng -> [5, 1, 1]
Lỗi 3 if ở đuôi      nums=[3, 1, 5]                   k=3: sai -> [3]                        đúng -> [5]
Lỗi 4 lưu giá trị    nums=[3, 3, 1]                   k=2: sai -> [3, 1]                     đúng -> [3, 3]
Lỗi 5 ghi sớm        nums=[1, 3, -1, -3, 5, 3, 6, 7]  k=3: sai -> [1, 3, 3, 3, 5, 5, 6, 7]   đúng -> [3, 3, 5, 5, 6, 7]
Lỗi 6 heap dùng if   nums=[8, 9, 1, 1]                k=2: sai -> [9, 9, 8]                  đúng -> [9, 9, 1]


## 12. Mẹo nhận dạng pattern và các bài liên quan

### Dấu hiệu nên nghĩ tới monotonic deque

- Đề hỏi **max / min của mọi đoạn con liền nhau** có độ dài cố định `k` (hoặc thoả một điều kiện), và cửa sổ **trượt một chiều** (hai đầu chỉ tiến, không lùi).
- Brute force có dạng "với mỗi vị trí, quét lại `k` phần tử gần nhất để lấy max/min" → gần như chắc chắn thay được bằng monotonic deque.
- **DP** có công thức chuyển trạng thái kiểu `dp[i] = nums[i] + max(dp[i-k .. i-1])` → "max của cửa sổ trượt trên mảng `dp`" → deque biến `O(n·k)` thành `O(n)` (bài tập 6).
- Cần cả max **và** min của cửa sổ → **hai** deque song song: một giảm dần (cho max), một tăng dần (cho min) (bài tập 5).

### Đổi max thành min

Chỉ cần **đảo chiều so sánh** ở bước đuôi: `while dq and nums[dq[-1]] >= x` → deque có giá trị **tăng dần**, đầu là min. (Bài tập 4.)

### Phân biệt với monotonic **stack**

| | Monotonic stack | Monotonic deque |
|---|---|---|
| Thao tác | chỉ ở **một đầu** (đỉnh) | ở **hai đầu** |
| Câu hỏi điển hình | "phần tử lớn hơn / nhỏ hơn **gần nhất**" | "max / min của **cửa sổ trượt**" |
| Vì sao cần đầu thứ hai | không cần | để bỏ phần tử **hết hạn** khi cửa sổ trượt |

Có thể coi deque = monotonic stack (đầu đuôi) + thêm khả năng "hết hạn" ở đầu kia.

### Câu thần chú

> **"Đầu hết hạn — Đuôi vô vọng — Thêm mới — Ghi đầu."**

### Bài liên quan (luyện tiếp)

| Bài | Liên hệ |
|---|---|
| 1438. Longest Continuous Subarray With Absolute Diff Less Than or Equal to Limit (Medium) | Cửa sổ **co giãn** + hai deque max/min (Bài tập 5) |
| 1696. Jump Game VI (Medium) | DP + max cửa sổ trượt trên mảng `dp` (Bài tập 6) |
| 1425. Constrained Subsequence Sum (Hard) | Giống 1696, thêm lựa chọn "bắt đầu lại": `dp[i] = nums[i] + max(0, max(dp[i-k..i-1]))` |
| 862. Shortest Subarray with Sum at Least K (Hard) | Monotonic deque trên **prefix sum** (có số âm nên không dùng hai con trỏ thường được) |
| 1499. Max Value of Equation (Hard) | Biến đổi `yi + yj + |xi - xj|` thành max cửa sổ của `y - x` |
| 2398. Maximum Number of Robots Within Budget (Hard) | Cửa sổ co giãn + deque cho max + tổng chạy |
| 480. Sliding Window Median (Hard) | Cùng khung cửa sổ trượt, nhưng cần **trung vị** → hai heap + lazy deletion (cách 2!) |
| 84. Largest Rectangle in Histogram (Hard) | Họ hàng gần: monotonic **stack** |
| 739. Daily Temperatures (Medium), 496. Next Greater Element I (Easy) | Nhập môn monotonic stack, nên học song song |

## 13. Bài tập vận dụng

Sáu bài, tăng dần độ khó. Hãy tự làm trước, bí thì xem **Gợi ý theo mức** (mục 14), cuối cùng mới mở **Đáp án** (mục 15). Các cell TODO chỉ định nghĩa hàm, **không** gọi hàm; test để sẵn dưới dạng comment — làm xong thì bỏ dấu `#` để tự kiểm tra.

### Bài 1 — Đoán output và trạng thái deque (không chạy code)

Với `nums = [4, 2, 12, 11, -5]`, `k = 2`, hãy chạy tay cách 4 và điền bảng:

| i | x | ① đầu hết hạn? | ② đuôi bỏ những ai? | deque (chỉ số) sau bước 3 | max ghi được |
|---:|---:|---|---|---|---:|
| 0 | 4 | ? | ? | ? | ? |
| 1 | 2 | ? | ? | ? | ? |
| 2 | 12 | ? | ? | ? | ? |
| 3 | 11 | ? | ? | ? | ? |
| 4 | -5 | ? | ? | ? | ? |

Kết quả cuối cùng là list nào? Ở bước nào nhánh "hết hạn" chạy?

### Bài 2 — Tính tay cách chia khối

Với `nums = [2, 7, 3, 8, 1, 4, 6]`, `k = 3`:

1. Chia khối và viết ra `prefix_max`, `suffix_max` (nhớ khối cuối chỉ có 1 phần tử).
2. Tính `max(suffix_max[s], prefix_max[s + k - 1])` cho `s = 0..4`, rồi đối chiếu với max từng cửa sổ tính bằng mắt.

### Bài 3 — Điền TODO: monotonic deque

Hoàn thiện cách 4 theo 4 bước. Đừng nhìn lại code ở trên — hãy dựa vào câu thần chú.

In [31]:
def exercise_max_sliding_window(nums: List[int], k: int) -> List[int]:
    dq = deque()            # lưu CHỈ SỐ, giá trị giảm thật sự từ đầu -> cuối
    result = []
    for i, x in enumerate(nums):
        # TODO 1: nếu chỉ số ở đầu deque đã rời cửa sổ (<= i - k) thì popleft
        # TODO 2: while deque còn phần tử và giá trị ở cuối <= x: pop
        # TODO 3: thêm i vào cuối deque
        # TODO 4: nếu i >= k - 1 thì ghi nums[dq[0]] vào result
        pass
    return result

# Tự làm xong rồi mới mở test:
# assert exercise_max_sliding_window([1, 3, -1, -3, 5, 3, 6, 7], 3) == [3, 3, 5, 5, 6, 7]
# assert exercise_max_sliding_window([1], 1) == [1]
# assert exercise_max_sliding_window([9, 5, 7, 3, 1, 4, 4, 2], 3) == [9, 7, 7, 4, 4, 4]

### Bài 4 — Biến thể: Sliding Window **Minimum**

Viết hàm trả về **min** của mọi cửa sổ độ dài `k`. Yêu cầu `O(n)`.

```text
nums = [1, 3, -1, -3, 5, 3, 6, 7], k = 3  ->  [-1, -3, -3, -3, 3, 3]
nums = [4, 2, 12, 11, -5], k = 2          ->  [2, 2, 11, -5]
```

Câu hỏi phụ: deque của bạn có giá trị tăng hay giảm từ đầu tới cuối? Phần tử "vô vọng" bây giờ được định nghĩa thế nào?

In [32]:
def exercise_min_sliding_window(nums: List[int], k: int) -> List[int]:
    # TODO: giống cách 4, chỉ đổi chiều một phép so sánh
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_min_sliding_window([1, 3, -1, -3, 5, 3, 6, 7], 3) == [-1, -3, -3, -3, 3, 3]
# assert exercise_min_sliding_window([4, 2, 12, 11, -5], 2) == [2, 2, 11, -5]
# assert exercise_min_sliding_window([5], 1) == [5]

### Bài 5 — Biến thể: LeetCode 1438 — Longest Continuous Subarray With Absolute Diff Less Than or Equal to Limit (Medium)

Cho mảng số nguyên `nums` và số nguyên `limit`, trả về **độ dài lớn nhất** của một mảng con liền nhau, khác rỗng, sao cho hiệu tuyệt đối giữa **hai phần tử bất kỳ** trong mảng con đó **không vượt quá** `limit`.

```python
class Solution:
    def longestSubarray(self, nums: List[int], limit: int) -> int:
```

```text
nums = [8,2,4,7], limit = 4            -> 2   ([2,4] hoặc [4,7])
nums = [10,1,2,4,7,2], limit = 5       -> 4   ([2,4,7,2]: max 7 - min 2 = 5)
nums = [4,2,2,2,4,4,2,2], limit = 0    -> 3   ([2,2,2])
```

Constraints: `1 <= nums.length <= 10^5`, `1 <= nums[i] <= 10^9`, `0 <= limit <= 10^9`.

Câu hỏi tư duy: "hai phần tử bất kỳ chênh nhau không quá `limit`" tương đương với điều kiện gì về **max** và **min** của mảng con? Lần này cửa sổ **không có độ dài cố định** — đầu phải tiến từng bước, đầu trái chỉ tiến khi cửa sổ vi phạm điều kiện. Vì sao cách 3 (chia khối) không dùng được ở đây, còn deque thì được?

In [33]:
def exercise_longest_subarray(nums: List[int], limit: int) -> int:
    # TODO 1: hai deque: max_dq (giá trị giảm dần), min_dq (giá trị tăng dần); left = 0; best = 0
    # TODO 2: với mỗi right: đưa right vào cả hai deque (bỏ phần tử vô vọng ở đuôi trước)
    # TODO 3: while nums[max_dq[0]] - nums[min_dq[0]] > limit: tăng left, bỏ chỉ số < left ở đầu mỗi deque
    # TODO 4: best = max(best, right - left + 1)
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_longest_subarray([8, 2, 4, 7], 4) == 2
# assert exercise_longest_subarray([10, 1, 2, 4, 7, 2], 5) == 4
# assert exercise_longest_subarray([4, 2, 2, 2, 4, 4, 2, 2], 0) == 3

### Bài 6 — Mini challenge: LeetCode 1696 — Jump Game VI (Medium)

Cho mảng `nums` (đánh chỉ số từ 0) và số nguyên `k`. Ban đầu bạn đứng ở chỉ số `0`. Mỗi bước, từ chỉ số `i` bạn có thể nhảy **tối đa `k` bước** về phía trước, tức tới bất kỳ chỉ số nào trong `[i + 1, min(n - 1, i + k)]`. Bạn muốn tới chỉ số cuối `n - 1`. **Điểm** của bạn là tổng `nums[j]` của mọi chỉ số `j` bạn đã đứng (kể cả đầu và cuối). Trả về **điểm lớn nhất** có thể đạt.

```python
class Solution:
    def maxResult(self, nums: List[int], k: int) -> int:
```

```text
nums = [1,-1,-2,4,-7,3], k = 2               -> 7    (đi 1 -> -1 -> 4 -> 3)
nums = [10,-5,-2,4,0,3], k = 3               -> 17   (đi 10 -> 4 -> 3)
nums = [1,-5,-20,4,-1,3,-6,-3], k = 2        -> 0
```

Constraints: `1 <= nums.length, k <= 10^5`, `-10^4 <= nums[i] <= 10^4`.

Câu hỏi tư duy: gọi `dp[i]` là điểm lớn nhất khi **đứng tại** `i`. Viết công thức `dp[i]` theo các `dp[j]` trước đó. Cách tính thẳng tốn `O(n·k)` — phần nào của công thức chính là "max của cửa sổ trượt"?

In [34]:
def exercise_max_result(nums: List[int], k: int) -> int:
    # TODO 1: dp = [0] * n; dp[0] = nums[0]; dq = deque([0]) chứa chỉ số, dp giảm dần
    # TODO 2: với i = 1..n-1: bỏ ở đầu deque các chỉ số < i - k (không nhảy tới i được nữa)
    # TODO 3: dp[i] = nums[i] + dp[dq[0]]
    # TODO 4: bỏ ở đuôi các chỉ số có dp <= dp[i], rồi append(i)
    # TODO 5: trả về dp[n - 1]
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_max_result([1, -1, -2, 4, -7, 3], 2) == 7
# assert exercise_max_result([10, -5, -2, 4, 0, 3], 3) == 17
# assert exercise_max_result([1, -5, -20, 4, -1, 3, -6, -3], 2) == 0

## 14. Gợi ý theo mức

Mỗi bài có 3 mức: **(1)** hướng tư duy, **(2)** cấu trúc / cú pháp, **(3)** gần như code. Chỉ mở mức tiếp theo khi mức trước chưa đủ.

**Bài 1**
1. Mỗi dòng làm đúng 4 bước theo thứ tự: đầu → đuôi → thêm → ghi. Với `k = 2`, ngưỡng hết hạn là `i - 2`.
2. Nhánh "hết hạn" chỉ chạy khi `dq[0] == i - 2`. Hãy để ý lúc `12` vào và lúc `-5` vào.
3. Ở `i = 2`: trước bước 1 deque là `[0, 1]`; `0 <= 0` nên `popleft`; rồi `12` đuổi `2`.

**Bài 2**
1. Khối là `[2, 7, 3] | [8, 1, 4] | [6]`. `prefix_max` reset ở chỉ số `0, 3, 6`; `suffix_max` reset ở chỉ số `2, 5, 6`.
2. `prefix_max` chạy trái → phải trong khối, `suffix_max` chạy phải → trái trong khối.
3. `s = 1`: phần trái là `[7, 3]` → `suffix_max[1] = 7`; phần phải là `[8]` → `prefix_max[3] = 8`.

**Bài 3**
1. Đọc lại câu thần chú: "Đầu hết hạn — Đuôi vô vọng — Thêm mới — Ghi đầu".
2. Bước 1 dùng `if dq and dq[0] <= i - k`, bước 2 dùng `while dq and nums[dq[-1]] <= x`.
3. Bốn dòng lệnh chính: `dq.popleft()`, `dq.pop()`, `dq.append(i)`, `result.append(nums[dq[0]])`.

**Bài 4**
1. Với min, phần tử cũ **lớn hơn hoặc bằng** phần tử mới là vô vọng (mới nhỏ hơn và ở lâu hơn).
2. Deque giữ giá trị **tăng dần** từ đầu tới cuối; đầu là min.
3. Chỉ đổi đúng một dòng: `while dq and nums[dq[-1]] >= x: dq.pop()`.

**Bài 5**
1. "Hai phần tử bất kỳ chênh không quá `limit`" ⇔ `max(đoạn) - min(đoạn) <= limit`. Nên cần biết max **và** min của cửa sổ hiện tại.
2. Dùng kỹ thuật **hai con trỏ** `left`, `right`: `right` tiến mỗi vòng; nếu vi phạm thì tăng `left` cho tới khi hợp lệ. Mỗi lần tăng `left`, chỉ số `left - 1` rời cửa sổ — bỏ nó khỏi đầu deque nào có nó.
3. Trong vòng `while vi phạm`: `left += 1`, rồi `if max_dq[0] < left: max_dq.popleft()` và tương tự cho `min_dq`.

**Bài 6**
1. Muốn đứng ở `i` thì bước trước phải đứng ở một `j` trong `[i - k, i - 1]`. Vậy `dp[i] = nums[i] + max(dp[i-k .. i-1])`.
2. `max(dp[i-k .. i-1])` là max của một cửa sổ trượt độ dài `k` trên mảng `dp` — dùng monotonic deque **trên `dp`** (so sánh `dp[...]`, không phải `nums[...]`).
3. Thứ tự trong mỗi vòng: bỏ đầu hết hạn (`dq[0] < i - k`) → `dp[i] = nums[i] + dp[dq[0]]` → bỏ đuôi `dp[dq[-1]] <= dp[i]` → `dq.append(i)`. Chú ý: phải tính xong `dp[i]` rồi mới đưa `i` vào deque.

## 15. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

**Bài 1** — `nums = [4, 2, 12, 11, -5]`, `k = 2`:

| i | x | ① đầu hết hạn? | ② đuôi bỏ những ai? | deque (chỉ số) sau bước 3 | max ghi được |
|---:|---:|---|---|---|---:|
| 0 | 4 | — | — | `[0]` | — (chưa đủ k) |
| 1 | 2 | — (`0 <= -1` sai) | — (`4 > 2`) | `[0, 1]` | 4 |
| 2 | 12 | **bỏ `0`** (`0 <= 0`) | bỏ `1` (giá trị 2) | `[2]` | 12 |
| 3 | 11 | — (`2 <= 1` sai) | — (`12 > 11`) | `[2, 3]` | 12 |
| 4 | -5 | **bỏ `2`** (`2 <= 2`) | — (`11 > -5`) | `[3, 4]` | 11 |

Kết quả `[4, 12, 12, 11]`. Nhánh hết hạn chạy ở `i = 2` và `i = 4`.

**Bài 2** — khối `[2, 7, 3] | [8, 1, 4] | [6]`:

| i | 0 | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---:|---:|---:|---:|---:|---:|---:|
| `nums` | 2 | 7 | 3 | 8 | 1 | 4 | 6 |
| `prefix_max` | 2 | 7 | 7 | 8 | 8 | 8 | 6 |
| `suffix_max` | 7 | 7 | 3 | 8 | 4 | 4 | 6 |

`s = 0`: `max(7, 7) = 7`; `s = 1`: `max(7, 8) = 8`; `s = 2`: `max(3, 8) = 8`; `s = 3`: `max(8, 8) = 8` (trùng khít khối 1); `s = 4`: `max(4, 6) = 6`. Kết quả `[7, 8, 8, 8, 6]`, khớp với max từng cửa sổ `[2,7,3]=7, [7,3,8]=8, [3,8,1]=8, [8,1,4]=8, [1,4,6]=6`.

Cell dưới kiểm chứng cả hai bài bằng code đã có.

In [35]:
# Kiểm chứng Bài 1
print("Bài 1:")
assert trace_deque([4, 2, 12, 11, -5], 2) == [4, 12, 12, 11]

# Kiểm chứng Bài 2
pre, suf = build_block_max([2, 7, 3, 8, 1, 4, 6], 3)
assert pre == [2, 7, 7, 8, 8, 8, 6]
assert suf == [7, 7, 3, 8, 4, 4, 6]
assert max_sliding_window_blocks([2, 7, 3, 8, 1, 4, 6], 3) == [7, 8, 8, 8, 6]
print("\nBài 2: prefix_max =", pre, "| suffix_max =", suf, "-> PASS")

Bài 1:
i=0 x= 4 | đầu: -         | đuôi: -              | dq=[0]        giá trị=[4]         
i=1 x= 2 | đầu: -         | đuôi: -              | dq=[0, 1]     giá trị=[4, 2]       -> max=4
i=2 x=12 | đầu: bỏ 0(4)   | đuôi: 1(2)           | dq=[2]        giá trị=[12]         -> max=12
i=3 x=11 | đầu: -         | đuôi: -              | dq=[2, 3]     giá trị=[12, 11]     -> max=12
i=4 x=-5 | đầu: bỏ 2(12)  | đuôi: -              | dq=[3, 4]     giá trị=[11, -5]     -> max=11

Bài 2: prefix_max = [2, 7, 7, 8, 8, 8, 6] | suffix_max = [7, 7, 3, 8, 4, 4, 6] -> PASS


In [36]:
# Đáp án Bài 3
def answer_max_sliding_window(nums: List[int], k: int) -> List[int]:
    dq = deque()
    result = []
    for i, x in enumerate(nums):
        if dq and dq[0] <= i - k:          # đầu hết hạn
            dq.popleft()
        while dq and nums[dq[-1]] <= x:    # đuôi vô vọng
            dq.pop()
        dq.append(i)                       # thêm mới
        if i >= k - 1:                     # ghi đầu
            result.append(nums[dq[0]])
    return result

assert answer_max_sliding_window([1, 3, -1, -3, 5, 3, 6, 7], 3) == [3, 3, 5, 5, 6, 7]
assert answer_max_sliding_window([1], 1) == [1]
assert answer_max_sliding_window([9, 5, 7, 3, 1, 4, 4, 2], 3) == [9, 7, 7, 4, 4, 4]


# Đáp án Bài 4
def answer_min_sliding_window(nums: List[int], k: int) -> List[int]:
    dq = deque()                           # giá trị TĂNG thật sự từ đầu -> cuối, đầu là min
    result = []
    for i, x in enumerate(nums):
        if dq and dq[0] <= i - k:
            dq.popleft()
        while dq and nums[dq[-1]] >= x:    # chỉ đổi <= thành >=
            dq.pop()
        dq.append(i)
        if i >= k - 1:
            result.append(nums[dq[0]])
    return result

assert answer_min_sliding_window([1, 3, -1, -3, 5, 3, 6, 7], 3) == [-1, -3, -3, -3, 3, 3]
assert answer_min_sliding_window([4, 2, 12, 11, -5], 2) == [2, 2, 11, -5]
assert answer_min_sliding_window([5], 1) == [5]
random.seed(4)
for _ in range(500):
    arr = [random.randint(-5, 5) for _ in range(random.randint(1, 12))]
    k = random.randint(1, len(arr))
    assert answer_min_sliding_window(arr, k) == [min(arr[s:s + k]) for s in range(len(arr) - k + 1)]
print("Bài 3, Bài 4 PASS (Bài 4 đối chiếu brute force trên 500 mảng ngẫu nhiên)")

Bài 3, Bài 4 PASS (Bài 4 đối chiếu brute force trên 500 mảng ngẫu nhiên)


**Bài 5 — LeetCode 1438.** `max - min <= limit` là điều kiện của cả đoạn. Với mỗi `right`, ta tìm `left` nhỏ nhất sao cho đoạn `left..right` hợp lệ. Khi `right` tăng, `left` **không bao giờ cần lùi** (đoạn đã vi phạm thì mở rộng thêm càng vi phạm), nên `left` chỉ tiến → tổng cộng `O(n)` bước. Hai deque cho max và min của cửa sổ hiện tại trong `O(1)`. Cách chia khối không áp dụng được vì độ dài cửa sổ thay đổi liên tục. Time `O(n)`, Space `O(n)`.

**Bài 6 — LeetCode 1696.** `dp[i] = nums[i] + max(dp[i-k .. i-1])`, `dp[0] = nums[0]`. Phần `max(...)` là max cửa sổ trượt độ dài `k` trên mảng `dp`, mà `dp` được tính **dần từ trái sang phải** — đúng kiểu dữ liệu "online" mà monotonic deque xử lý được (cách chia khối thì không, vì cần biết trước các `dp` bên phải). Time `O(n)`, Space `O(n)` cho `dp` (có thể giảm, nhưng không cần).

In [37]:
# Đáp án Bài 5
def answer_longest_subarray(nums: List[int], limit: int) -> int:
    max_dq, min_dq = deque(), deque()      # max_dq: giá trị giảm dần; min_dq: giá trị tăng dần
    left = best = 0
    for right, x in enumerate(nums):
        while max_dq and nums[max_dq[-1]] <= x:
            max_dq.pop()
        max_dq.append(right)
        while min_dq and nums[min_dq[-1]] >= x:
            min_dq.pop()
        min_dq.append(right)
        while nums[max_dq[0]] - nums[min_dq[0]] > limit:   # vi phạm -> thu hẹp từ trái
            left += 1
            if max_dq[0] < left:
                max_dq.popleft()
            if min_dq[0] < left:
                min_dq.popleft()
        best = max(best, right - left + 1)
    return best

def longest_subarray_bruteforce(nums, limit):
    best = 0
    for i in range(len(nums)):
        hi = lo = nums[i]
        for j in range(i, len(nums)):
            hi, lo = max(hi, nums[j]), min(lo, nums[j])
            if hi - lo <= limit:
                best = max(best, j - i + 1)
    return best

assert answer_longest_subarray([8, 2, 4, 7], 4) == 2
assert answer_longest_subarray([10, 1, 2, 4, 7, 2], 5) == 4
assert answer_longest_subarray([4, 2, 2, 2, 4, 4, 2, 2], 0) == 3
random.seed(1438)
for _ in range(500):
    arr = [random.randint(1, 8) for _ in range(random.randint(1, 12))]
    lim = random.randint(0, 7)
    assert answer_longest_subarray(arr, lim) == longest_subarray_bruteforce(arr, lim), (arr, lim)
print("Bài 5 PASS (đối chiếu brute force trên 500 mảng ngẫu nhiên)")


# Đáp án Bài 6
def answer_max_result(nums: List[int], k: int) -> int:
    n = len(nums)
    dp = [0] * n
    dp[0] = nums[0]
    dq = deque([0])                         # chỉ số, dp giảm thật sự từ đầu -> cuối
    for i in range(1, n):
        if dq[0] < i - k:                   # từ dq[0] không nhảy tới i được nữa
            dq.popleft()
        dp[i] = nums[i] + dp[dq[0]]         # max(dp[i-k .. i-1]) nằm ở đầu deque
        while dq and dp[dq[-1]] <= dp[i]:
            dq.pop()
        dq.append(i)
    return dp[-1]

def max_result_bruteforce(nums, k):
    n = len(nums)
    dp = [0] * n
    dp[0] = nums[0]
    for i in range(1, n):
        dp[i] = nums[i] + max(dp[max(0, i - k):i])
    return dp[-1]

assert answer_max_result([1, -1, -2, 4, -7, 3], 2) == 7
assert answer_max_result([10, -5, -2, 4, 0, 3], 3) == 17
assert answer_max_result([1, -5, -20, 4, -1, 3, -6, -3], 2) == 0
random.seed(1696)
for _ in range(500):
    arr = [random.randint(-10, 10) for _ in range(random.randint(1, 12))]
    k = random.randint(1, 6)
    assert answer_max_result(arr, k) == max_result_bruteforce(arr, k), (arr, k)
print("Bài 6 PASS (đối chiếu DP O(n·k) trên 500 mảng ngẫu nhiên)")

Bài 5 PASS (đối chiếu brute force trên 500 mảng ngẫu nhiên)
Bài 6 PASS (đối chiếu DP O(n·k) trên 500 mảng ngẫu nhiên)


## 16. Template ngắn nên nhớ

```python
from collections import deque

def sliding_window_max(nums, k):
    dq, res = deque(), []                 # dq: CHỈ SỐ, giá trị giảm dần
    for i, x in enumerate(nums):
        if dq and dq[0] <= i - k:         # 1. Đầu hết hạn
            dq.popleft()
        while dq and nums[dq[-1]] <= x:   # 2. Đuôi vô vọng   (min: đổi thành >=)
            dq.pop()
        dq.append(i)                      # 3. Thêm mới
        if i >= k - 1:                    # 4. Ghi đầu
            res.append(nums[dq[0]])
    return res
```

**Ba ý phải nhớ:**

1. Phần tử "trẻ hơn mà giỏi hơn (hoặc bằng)" làm phần tử cũ **vô vọng vĩnh viễn** → vứt khỏi đuôi.
2. Deque lưu **chỉ số** để biết khi nào **hết hạn**: chỉ số `<= i - k` đã rời cửa sổ kết thúc tại `i`.
3. Mỗi chỉ số vào 1 lần, ra tối đa 1 lần → **amortized `O(n)`**, Space `O(k)`.

## Checklist trước khi submit

- [ ] Kết quả có đúng `n - k + 1` phần tử? (chỉ ghi khi `i >= k - 1`)
- [ ] Deque lưu **chỉ số**, không phải giá trị?
- [ ] Điều kiện hết hạn là `dq[0] <= i - k` (không phải `<`)?
- [ ] Bước đuôi dùng `while` (không phải `if`) và có kiểm tra `dq` khác rỗng trước khi đọc `dq[-1]`?
- [ ] Chiều so sánh đúng: max → bỏ `<= x`, deque giảm dần; min → bỏ `>= x`, deque tăng dần?
- [ ] Dùng `collections.deque` với `popleft()`, không dùng `list.pop(0)`?
- [ ] Đã thử `k = 1`, `k = n`, mảng giảm dần (max liên tục hết hạn), mảng có số trùng, mảng toàn số âm?
- [ ] Nếu dùng heap: đổi dấu khi push **và** khi đọc max; dọn đỉnh bằng `while`?
- [ ] Đúng signature `maxSlidingWindow(self, nums: List[int], k: int) -> List[int]`, không còn `print` debug?